# Assignment 04 · Miền `customer_comments` — Notebook 01
# Mạng nơ-ron tích chập một chiều (1D CNN) hiện thực bằng **NumPy thuần**

- **Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
- **Học phần:** Phát triển các Hệ thống Thông minh · **Giảng viên:** PGS.TS Trần Đình Quế
- **Học kỳ:** Học kỳ 1 năm học 2026 – 2027
- **Assignment 04:** Convolutional Neural Networks — From Mathematical Convolution to NumPy, PyTorch and TensorFlow
- **Miền dữ liệu:** `customer_comments` — bình luận đánh giá sản phẩm thời trang nữ (Women's E-Commerce Clothing Reviews)

---

## 1. Mục tiêu của notebook

Notebook này là **notebook chủ đạo** của miền `customer_comments`. Mục tiêu không dừng ở việc đạt
một con số độ chính xác cao, mà là chứng minh rằng người viết nắm được **toàn bộ chuỗi tính toán**
của một mạng tích chập một chiều dùng cho phân loại quan điểm văn bản, mà không mượn bất kỳ thư
viện học sâu nào.

Cụ thể, báo cáo sẽ lần lượt:

1. Khảo sát tập dữ liệu 23 486 bình luận sản phẩm thời trang nữ và nhãn `Recommended IND`.
2. Trình bày **định nghĩa toán học của phép tích chập rời rạc** và kiểm chứng nó bằng một ví dụ số
   được tính tay từng bước, sau đó đối chiếu với kết quả do mã nguồn sinh ra.
3. Hiện thực từ đầu các tầng `Embedding`, `Conv1D`, `ReLU`, `GlobalMaxPool1D`, `Dense`, `Sigmoid`,
   hàm mất mát `BCE` và thuật toán tối ưu `Adam`, mỗi tầng là một lớp Python có hai phương thức
   `forward` và `backward`.
4. **Chứng minh tính đúng đắn của lượt truyền ngược** bằng phép kiểm tra gradient theo sai phân hữu
   hạn. Đây là phần có giá trị học thuật cao nhất của notebook: nó cho thấy các công thức đạo hàm
   được dẫn ra bằng tay là chính xác, chứ không phải được chép lại.
5. Huấn luyện, đánh giá và lưu chỉ số ra tệp JSON theo đúng hợp đồng tích hợp.
6. Tổng hợp kết quả của cả ba cách hiện thực (NumPy / PyTorch / TensorFlow) thành các hình so sánh.

### Kiến trúc mô hình (theo `CONTRACT.md`, Mục 4)

$$
\text{tokens}(50) \;\to\; \text{Embedding}(5000, 100) \;\to\; \text{Conv1D}(32, K{=}3) \;\to\;
\text{ReLU} \;\to\; \text{GlobalMaxPool1D} \;\to\; \text{Dense}(1) \;\to\; \sigma
$$

### Một lưu ý về hiệu năng

Toàn bộ phép tích chập được vector hóa bằng `np.lib.stride_tricks.sliding_window_view` kết hợp
`np.einsum`. Notebook **không** chứa vòng lặp Python duyệt theo vị trí cửa sổ; đây là điều kiện bắt
buộc để huấn luyện NumPy thuần hoàn tất trong ngân sách thời gian mà hợp đồng cho phép.


## 2. Nhập thư viện và cố định seed

Hợp đồng quy định `RANDOM_SEED = 42` ở mọi nơi. Việc cố định seed bảo đảm mọi con số xuất hiện
trong báo cáo đều có thể tái lập chính xác khi chạy lại notebook.

In [1]:
import os, re, json, time, math, collections
import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

# Cấu hình hình vẽ theo Mục 5.2 của hợp đồng
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

# Siêu tham số cố định bởi hợp đồng
VOCAB_SIZE = 5000
MAX_LEN    = 50
EMBED_DIM  = 100
N_FILTERS  = 32
KERNEL     = 3

DATA_PATH  = "../data/womens_ecommerce_reviews.csv"
FIG_DIR    = "../reports/figures"
REPORT_DIR = "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("NumPy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("Seed       :", RANDOM_SEED)

NumPy      : 2.5.1
pandas     : 3.0.5
matplotlib : 3.11.1
Seed       : 42


## 3. Nạp dữ liệu và khảo sát

Tập dữ liệu *Women's E-Commerce Clothing Reviews* gồm 23 486 bản ghi. Trường `Review Text` chứa nội
dung bình luận tự do của khách hàng, trường `Recommended IND` là nhãn nhị phân: giá trị 1 nghĩa là
khách hàng **có** giới thiệu sản phẩm, giá trị 0 nghĩa là **không**.

In [2]:
df_raw = pd.read_csv(DATA_PATH)
N_RAW = len(df_raw)
print("Kích thước bảng gốc :", df_raw.shape)
print("Các cột             :", list(df_raw.columns))
print()
print("Số giá trị khuyết theo cột:")
print(df_raw.isna().sum().to_string())

Kích thước bảng gốc : (23486, 11)
Các cột             : ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']

Số giá trị khuyết theo cột:
Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14


**Diễn giải.** Bảng gốc có 23 486 dòng và 11 cột. Cột `Review Text` thiếu 845 giá trị, tương ứng
3.60% số bản ghi; cột `Title` thiếu tới 3 810 giá trị nhưng không được dùng trong bài toán này nên
không ảnh hưởng. Đáng chú ý là cột nhãn `Recommended IND` **không thiếu giá trị nào**, nghĩa là toàn
bộ tổn thất dữ liệu ở bước làm sạch chỉ đến từ những bản ghi khách hàng chấm điểm nhưng không viết
nội dung bình luận. Ba cột `Division Name`, `Department Name`, `Class Name` cùng thiếu đúng 14 giá
trị, gợi ý rằng 14 bản ghi đó thiếu thông tin danh mục sản phẩm một cách đồng thời; chúng không nằm
trong phạm vi sử dụng của mô hình văn bản.

In [3]:
# Bước 1 của hợp đồng: loại bỏ bản ghi thiếu nội dung hoặc thiếu nhãn
df = df_raw.dropna(subset=["Review Text", "Recommended IND"]).reset_index(drop=True)
N_CLEAN = len(df)

labels = df["Recommended IND"].astype(int).values
n_pos  = int((labels == 1).sum())
n_neg  = int((labels == 0).sum())

print(f"Số bản ghi sau khi loại giá trị khuyết : {N_CLEAN:,} / {N_RAW:,} "
      f"(giữ lại {100*N_CLEAN/N_RAW:.2f}%)")
print(f"  Nhãn 1 (có giới thiệu)    : {n_pos:,}  ({100*n_pos/N_CLEAN:.2f}%)")
print(f"  Nhãn 0 (không giới thiệu) : {n_neg:,}  ({100*n_neg/N_CLEAN:.2f}%)")
print(f"  Tỉ lệ mất cân bằng        : {n_pos/n_neg:.2f} : 1")
print()
print("Ví dụ hai bình luận trái chiều:")
print("-" * 78)
print("[NHAN 1]", df.loc[df["Recommended IND"] == 1, "Review Text"].iloc[1][:240], "...")
print("-" * 78)
print("[NHAN 0]", df.loc[df["Recommended IND"] == 0, "Review Text"].iloc[0][:240], "...")

Số bản ghi sau khi loại giá trị khuyết : 22,641 / 23,486 (giữ lại 96.40%)
  Nhãn 1 (có giới thiệu)    : 18,540  (81.89%)
  Nhãn 0 (không giới thiệu) : 4,101  (18.11%)
  Tỉ lệ mất cân bằng        : 4.52 : 1

Ví dụ hai bình luận trái chiều:
------------------------------------------------------------------------------
[NHAN 1] Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.   ...
------------------------------------------------------------------------------
[NHAN 0] I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite  ...


**Diễn giải.** Sau khi loại bỏ bản ghi khuyết, tập dữ liệu còn 22 641 bình luận, giữ lại 96.40% dữ
liệu gốc. Phân phối nhãn **mất cân bằng rõ rệt**: 18 540 bình luận mang nhãn 1 (81.89%) so với 4 101
bình luận mang nhãn 0 (18.11%), tỉ lệ 4.52 : 1.

Con số này có hai hệ quả trực tiếp đối với phần đánh giá. Thứ nhất, một bộ phân loại tầm thường luôn
trả lời "có giới thiệu" đã đạt độ chính xác 81.89%; vì vậy mọi kết quả accuracy phải được đọc trong
sự so sánh với ngưỡng cơ sở này chứ không phải với mốc 50%. Thứ hai, precision và recall của lớp
thiểu số (nhãn 0) mới là thước đo thực sự cho biết mô hình có học được điều gì hay không, nên báo
cáo sẽ trình bày đầy đủ `classification_report` cho cả hai lớp thay vì chỉ một con số tổng hợp.

Hai bình luận trích dẫn ở trên minh họa vì sao bài toán có thể giải được bằng đặc trưng cục bộ: mẫu
nhãn 1 mở đầu bằng cụm "Love this dress", còn mẫu nhãn 0 chứa cụm "could not zip it up". Chính những
cụm ngắn như vậy là thứ mà một hạt nhân tích chập bề rộng 3 được thiết kế để phát hiện.

## 4. Tiền xử lý dữ liệu văn bản

Hợp đồng tích hợp (`CONTRACT.md`, Mục 3) quy định quy trình tiền xử lý **bắt buộc** cho miền
`customer_comments`, và báo cáo tuân thủ nguyên văn:

1. `dropna(subset=['Review Text','Recommended IND'])`: loại các bản ghi thiếu nội dung hoặc thiếu nhãn.
2. Hạ toàn bộ văn bản về chữ thường.
3. Tách token bằng biểu thức chính quy `re.findall(r'[a-zA-Z]+', text)`.
4. Từ điển kích thước `VOCAB_SIZE = 5000` với hai token đặc biệt `<PAD>` = 0 và `<UNK>` = 1.
5. Cắt hoặc đệm mọi chuỗi về độ dài cố định `MAX_LEN = 50`.
6. Chiều không gian nhúng `EMBED_DIM = 100`.

**Tại sao lại làm như vậy?** Báo cáo giải thích từng lựa chọn thay vì chỉ liệt kê thao tác:

- *Hạ chữ thường và chỉ giữ ký tự chữ cái.* Mục tiêu của bài toán là phân loại quan điểm, không phải
  phân tích chính tả. Nếu giữ nguyên dạng chữ thì `"Love"`, `"love"` và `"LOVE"` bị coi là ba token
  khác nhau, làm phân mảnh vô ích ngân sách 5 000 từ vựng. Biểu thức `[a-zA-Z]+` đồng thời loại bỏ
  chữ số, dấu câu và emoji, vốn là những thành phần mang rất ít thông tin quan điểm trong tập dữ liệu này
  nhưng lại chiếm chỗ trong từ điển.
- *Giới hạn từ điển ở 5 000 từ.* Ma trận nhúng có kích thước $|V| \times d$. Với $d = 100$, mỗi
  1 000 từ tăng thêm tương ứng 100 000 tham số. Giới hạn 5 000 giữ mô hình ở mức nửa triệu tham số,
  vừa đủ nhỏ để huấn luyện bằng NumPy thuần trên CPU trong ngân sách thời gian cho phép. Các từ
  hiếm rơi vào `<UNK>`; đây là hành vi mong muốn vì từ xuất hiện một hai lần không đủ dữ liệu để
  học được một vector nhúng có ý nghĩa.
- *Từ điển chỉ được xây trên tập huấn luyện.* Đây là điểm mấu chốt về tính trung thực của phép đánh
  giá. Nếu đếm tần suất trên toàn bộ dữ liệu, thông tin từ tập kiểm tra sẽ rò rỉ ngược vào giai đoạn
  thiết kế mô hình (data leakage) và chỉ số cuối cùng sẽ lạc quan hơn thực tế.
- *Đệm về độ dài cố định 50.* Phép tích chập một chiều được vector hóa theo lô đòi hỏi mọi mẫu trong
  lô có cùng độ dài. Token `<PAD>` = 0 được gán vector nhúng khởi tạo bằng 0 và **gradient của nó
  luôn bị ép về 0** sau mỗi bước, nên phần đệm không đóng góp tín hiệu học.


In [4]:
def tokenize(text: str):
    """Tách một chuỗi thành danh sách token theo đúng quy tắc của hợp đồng:
    hạ chữ thường rồi lấy mọi cụm ký tự chữ cái liên tiếp."""
    return re.findall(r"[a-zA-Z]+", text.lower())

t0 = time.time()
tokens_all = [tokenize(t) for t in df["Review Text"].astype(str)]
doc_lengths = np.array([len(t) for t in tokens_all])
print(f"Tách token xong trong {time.time()-t0:.2f}s")
print()
print("Thống kê độ dài bình luận (đơn vị: token)")
print(f"  trung bình    : {doc_lengths.mean():.2f}")
print(f"  trung vị      : {np.median(doc_lengths):.1f}")
print(f"  độ lệch chuẩn : {doc_lengths.std():.2f}")
print(f"  nhỏ nhất      : {doc_lengths.min()}   lớn nhất: {doc_lengths.max()}")
for q in [25, 50, 75, 90, 95]:
    print(f"  phân vị {q:>2}%   : {np.percentile(doc_lengths, q):.0f}")
print()
print(f"Tỉ lệ bình luận có độ dài <= MAX_LEN={MAX_LEN}: {100*(doc_lengths<=MAX_LEN).mean():.2f}%")
print(f"  -> {100*(doc_lengths>MAX_LEN).mean():.2f}% số bình luận bị cắt bớt phần đuôi.")

Tách token xong trong 1.58s

Thống kê độ dài bình luận (đơn vị: token)
  trung bình    : 60.77
  trung vị      : 60.0
  độ lệch chuẩn : 28.81
  nhỏ nhất      : 2   lớn nhất: 116
  phân vị 25%   : 36
  phân vị 50%   : 60
  phân vị 75%   : 89
  phân vị 90%   : 99
  phân vị 95%   : 102

Tỉ lệ bình luận có độ dài <= MAX_LEN=50: 40.76%
  -> 59.24% số bình luận bị cắt bớt phần đuôi.


**Diễn giải.** Độ dài trung bình của một bình luận là 60.77 token, trung vị 60 token, độ lệch chuẩn
28.81. Phân phối gần đối xứng quanh trung vị nhưng bị chặn trên ở 116 token, dấu hiệu cho thấy nguồn
dữ liệu đã giới hạn độ dài bình luận ngay từ khâu thu thập.

Quyết định `MAX_LEN = 50` của hợp đồng vì vậy **cắt bớt 59.24% số bình luận**, một tỉ lệ không hề nhỏ
và cần được nhìn nhận thẳng thắn. Có hai lý do khiến lựa chọn này vẫn hợp lý. Thứ nhất, chi phí tính
toán tăng tuyến tính theo độ dài chuỗi, nên nâng `MAX_LEN` lên 100 sẽ làm đôi thời gian huấn luyện
của phiên bản NumPy thuần. Thứ hai, và quan trọng hơn, kiến trúc kết thúc bằng `GlobalMaxPool1D`: mô
hình chỉ cần **một** cụm ba từ mang quan điểm xuất hiện đâu đó trong 50 token đầu là đủ để phân loại
đúng. Người viết đánh giá quan điểm thường được bộc lộ sớm, nên phần đuôi bị cắt chủ yếu chứa mô tả
chi tiết về kích cỡ và chất liệu. Phần thực nghiệm ở Mục 10 xác nhận suy luận này: mô hình vẫn đạt độ
chính xác 88.3% dù chỉ thấy một nửa số token.

### 4.1 Hình 1 — Khảo sát dữ liệu (`fig_comments_eda.png`)

Hình gồm hai bảng con theo yêu cầu của hợp đồng (Mục 6): bên trái là phân phối nhãn, bên phải là
biểu đồ tần suất độ dài bình luận kèm đường cắt tại `MAX_LEN = 50`.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# --- Bảng con trái: phân phối nhãn ---
ax = axes[0]
bars = ax.bar(["Không giới thiệu\n(nhãn 0)", "Có giới thiệu\n(nhãn 1)"],
              [n_neg, n_pos], color=["#c0392b", "#2980b9"], width=0.55)
for b, v in zip(bars, [n_neg, n_pos]):
    ax.text(b.get_x() + b.get_width()/2, v + 250,
            f"{v:,}\n({100*v/N_CLEAN:.1f}%)", ha="center", va="bottom", fontsize=11)
ax.set_title("Phân phối nhãn Recommended IND", fontsize=13, fontweight="bold")
ax.set_ylabel("Số lượng bình luận")
ax.set_ylim(0, n_pos * 1.20)
ax.grid(axis="y", alpha=0.3)

# --- Bảng con phải: biểu đồ tần suất độ dài ---
ax = axes[1]
ax.hist(doc_lengths, bins=60, color="#16a085", edgecolor="white", alpha=0.9)
ax.axvline(MAX_LEN, color="#c0392b", linestyle="--", linewidth=2,
           label=f"MAX_LEN = {MAX_LEN}")
ax.axvline(doc_lengths.mean(), color="#2c3e50", linestyle=":", linewidth=2,
           label=f"Trung bình = {doc_lengths.mean():.1f}")
ax.set_title("Phân phối độ dài bình luận (số token)", fontsize=13, fontweight="bold")
ax.set_xlabel("Số token trong một bình luận")
ax.set_ylabel("Số lượng bình luận")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_comments_eda.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_comments_eda.png")

Đã lưu: ../reports/figures/fig_comments_eda.png


![Khảo sát dữ liệu](../reports/figures/fig_comments_eda.png)

**Diễn giải hình 1.** Bảng con bên trái cho thấy độ nghiêng của phân phối nhãn một cách trực quan:
cột nhãn 1 cao gấp khoảng 4.5 lần cột nhãn 0 (18 540 so với 4 101). Đây chính là ngưỡng cơ sở 81.9%
mà mọi mô hình phải vượt qua một cách có ý nghĩa.

Bảng con bên phải cho thấy phân phối độ dài có dạng gần đều trong khoảng 20 tới 100 token với một
đỉnh nhọn ở sát biên phải, hệ quả của việc dữ liệu gốc bị giới hạn độ dài. Đường đứt nét đỏ tại
`MAX_LEN = 50` nằm **bên trái đường trung bình 60.8 token**, nghĩa là điểm cắt rơi vào vùng đông đúc
của phân phối chứ không phải ở phần đuôi thưa thớt. Quan sát này củng cố nhận định ở đoạn trên: việc
cắt chuỗi là một đánh đổi có ý thức giữa chi phí tính toán và lượng thông tin, chứ không phải một
thao tác vô hại.

### 4.2 Chia tập và xây từ điển

Báo cáo chia dữ liệu theo tỉ lệ **70% huấn luyện / 15% kiểm định / 15% kiểm tra**, có **phân tầng
theo nhãn** (`stratify`) để giữ nguyên tỉ lệ lớp trong cả ba tập, và `random_state=42` theo hợp đồng.
Tập kiểm định chỉ dùng để chọn epoch tốt nhất; tập kiểm tra **không** tham gia vào bất kỳ quyết định
thiết kế nào và chỉ được chạm tới đúng một lần ở bước đánh giá cuối cùng.

In [6]:
idx_all = np.arange(N_CLEAN)
idx_tmp, idx_test = train_test_split(idx_all, test_size=0.15,
                                     stratify=labels, random_state=RANDOM_SEED)
idx_train, idx_val = train_test_split(idx_tmp, test_size=0.1765,
                                      stratify=labels[idx_tmp], random_state=RANDOM_SEED)

N_TRAIN, N_VAL, N_TEST = len(idx_train), len(idx_val), len(idx_test)
print(f"Tập huấn luyện : {N_TRAIN:,} mẫu ({100*N_TRAIN/N_CLEAN:.1f}%) "
      f"- tỉ lệ nhãn 1: {labels[idx_train].mean():.4f}")
print(f"Tập kiểm định  : {N_VAL:,} mẫu ({100*N_VAL/N_CLEAN:.1f}%) "
      f"- tỉ lệ nhãn 1: {labels[idx_val].mean():.4f}")
print(f"Tập kiểm tra   : {N_TEST:,} mẫu ({100*N_TEST/N_CLEAN:.1f}%) "
      f"- tỉ lệ nhãn 1: {labels[idx_test].mean():.4f}")

Tập huấn luyện : 15,847 mẫu (70.0%) - tỉ lệ nhãn 1: 0.8188
Tập kiểm định  : 3,397 mẫu (15.0%) - tỉ lệ nhãn 1: 0.8190
Tập kiểm tra   : 3,397 mẫu (15.0%) - tỉ lệ nhãn 1: 0.8190


In [7]:
# Từ điển CHỈ được đếm trên tập huấn luyện để tránh rò rỉ dữ liệu
counter = collections.Counter(w for i in idx_train for w in tokens_all[i])
print(f"Số từ phân biệt trong tập huấn luyện: {len(counter):,}")

word2id = {"<PAD>": 0, "<UNK>": 1}
for w, _ in counter.most_common(VOCAB_SIZE - 2):
    word2id[w] = len(word2id)
id2word = {i: w for w, i in word2id.items()}
print(f"Kích thước từ điển thực tế: {len(word2id):,} (đã gồm <PAD> và <UNK>)")

cover = sum(c for w, c in counter.items() if w in word2id) / sum(counter.values())
print(f"Tỉ lệ token của tập huấn luyện được từ điển bao phủ: {100*cover:.2f}%")
print()
print("15 từ có tần suất cao nhất:", [w for w, _ in counter.most_common(15)])

def encode(token_list):
    """Ánh xạ token -> chỉ số, cắt ở MAX_LEN và đệm 0 ở phía sau."""
    X = np.zeros((len(token_list), MAX_LEN), dtype=np.int64)
    for i, toks in enumerate(token_list):
        ids = [word2id.get(w, 1) for w in toks[:MAX_LEN]]
        X[i, :len(ids)] = ids
    return X

X_train = encode([tokens_all[i] for i in idx_train])
X_val   = encode([tokens_all[i] for i in idx_val])
X_test  = encode([tokens_all[i] for i in idx_test])
y_train = labels[idx_train].astype(np.float64)
y_val   = labels[idx_val].astype(np.float64)
y_test  = labels[idx_test].astype(np.float64)

print()
print("Kích thước ma trận sau mã hóa:", X_train.shape, X_val.shape, X_test.shape)
unk_rate = (X_train == 1).sum() / (X_train != 0).sum()
pad_rate = (X_train == 0).mean()
print(f"Tỉ lệ token <UNK> trong tập huấn luyện : {100*unk_rate:.2f}%")
print(f"Tỉ lệ ô bị đệm <PAD> trong ma trận     : {100*pad_rate:.2f}%")
print()
print("Ví dụ một mẫu đã mã hóa (20 vị trí đầu):")
print(X_train[0][:20])
print("Giải mã ngược:", " ".join(id2word[i] for i in X_train[0][:20]))

Số từ phân biệt trong tập huấn luyện: 11,724
Kích thước từ điển thực tế: 5,000 (đã gồm <PAD> và <UNK>)
Tỉ lệ token của tập huấn luyện được từ điển bao phủ: 99.06%

15 từ có tần suất cao nhất: ['the', 'i', 'it', 'and', 'a', 'is', 'this', 'to', 'in', 'but', 'on', 'for', 'of', 'with', 'was']



Kích thước ma trận sau mã hóa: (15847, 50) (3397, 50) (3397, 50)
Tỉ lệ token <UNK> trong tập huấn luyện : 0.81%
Tỉ lệ ô bị đệm <PAD> trong ma trận     : 15.18%

Ví dụ một mẫu đã mã hóa (20 vị trí đầu):
[   3   53   23   60  267    2  243 2105   20   77    2   93   27  187
   88  628   98    7   38    3]
Giải mã ngược: i really love these leggings the different textures that all the colors have make them interesting quality is great i


**Diễn giải.** Tập huấn luyện chứa 11 724 từ phân biệt, nhưng từ điển chỉ giữ 5 000 từ phổ biến
nhất. Điều thoạt nhìn có vẻ là tổn thất lớn (loại bỏ 57% số từ) thực ra rất ít tốn kém: **99.06% số
lần xuất hiện token** vẫn được từ điển bao phủ, và tỉ lệ token bị thay bằng `<UNK>` trong ma trận
huấn luyện chỉ là 0.81%. Đây là biểu hiện kinh điển của định luật Zipf trong ngôn ngữ tự nhiên, theo
đó một số ít từ chiếm phần lớn tần suất xuất hiện.

Danh sách 15 từ tần suất cao nhất toàn là hư từ (`the`, `i`, `it`, `and`, `a`), không mang thông tin
quan điểm. Mô hình không bị nhiễu bởi điều này vì tầng `Conv1D` xét **cụm ba từ liên tiếp** chứ không
xét từ đơn lẻ: một hư từ chỉ đóng vai trò ngữ cảnh trong cụm, chẳng hạn cụm "and very flattering" mà
Mục 10.1 sẽ chỉ ra.

Tỉ lệ ô bị đệm `<PAD>` là 15.18%, phù hợp với con số 40.76% bình luận ngắn hơn 50 token đã tính ở
trên. Vì hàng `<PAD>` được ép bằng 0 và bị chặn gradient, 15.18% ô này không đóng góp vào quá trình
học.

Mẫu giải mã ngược ở cuối ô mã xác nhận quy trình mã hóa và giải mã là nghịch đảo của nhau: chuỗi chỉ
số khôi phục đúng câu "i really love these leggings the different textures ...", chứng tỏ từ điển và
hàm `encode` hoạt động đúng.

---

## 5. Cơ sở toán học của phép tích chập rời rạc

### 5.1 Định nghĩa

Cho tín hiệu một chiều $x \in \mathbb{R}^{L}$ và hạt nhân (kernel) $k \in \mathbb{R}^{K}$. Trong
giải tích tín hiệu, **tích chập** được định nghĩa là

$$
(x * k)[i] \;=\; \sum_{m} x[m]\, k[i-m].
$$

Tuy nhiên, hầu như mọi thư viện học sâu (PyTorch `nn.Conv1d`, Keras `Conv1D`) **không** lật hạt nhân
mà hiện thực phép **tương quan chéo** (cross-correlation):

$$
y[i] \;=\; \sum_{j=0}^{K-1} x[i+j]\, k[j], \qquad i = 0, 1, \dots, L-K .
$$

Sự khác biệt chỉ là một phép lật chỉ số của $k$. Vì $k$ là **tham số được học**, mạng có thể tự học
ra hạt nhân đã lật, nên hai công thức tương đương về khả năng biểu diễn. Báo cáo dùng công thức
tương quan chéo để kết quả NumPy khớp tuyệt đối với PyTorch và TensorFlow ở hai notebook sau.

Với chế độ `valid` (không đệm, bước nhảy $s = 1$), độ dài đầu ra là

$$
L_{\text{out}} \;=\; \left\lfloor \frac{L + 2p - K}{s} \right\rfloor + 1
\;\;\xrightarrow{\;p=0,\; s=1\;}\;\; L_{\text{out}} = L - K + 1 .
$$

Với $L = 50$ và $K = 3$, đầu ra có $L_{\text{out}} = 48$ vị trí.

### 5.2 Ví dụ số tính tay

Để chắc chắn rằng công thức trên được hiểu đúng trước khi viết mã, báo cáo tính tay một ví dụ cụ thể:

$$
x = [\,2,\; 1,\; 3,\; 4,\; 2\,], \qquad k = [\,0.5,\; -1,\; 0.5\,].
$$

Ở đây $L = 5$, $K = 3$, nên $L_{\text{out}} = 5 - 3 + 1 = 3$. Trượt cửa sổ ba phần tử dọc theo $x$:

**Vị trí $i = 0$**, cửa sổ $[2, 1, 3]$:

$$
y[0] = 0.5 \cdot 2 \;+\; (-1) \cdot 1 \;+\; 0.5 \cdot 3
     = 1.0 - 1.0 + 1.5 = \mathbf{1.5}
$$

**Vị trí $i = 1$**, cửa sổ $[1, 3, 4]$:

$$
y[1] = 0.5 \cdot 1 \;+\; (-1) \cdot 3 \;+\; 0.5 \cdot 4
     = 0.5 - 3.0 + 2.0 = \mathbf{-0.5}
$$

**Vị trí $i = 2$**, cửa sổ $[3, 4, 2]$:

$$
y[2] = 0.5 \cdot 3 \;+\; (-1) \cdot 4 \;+\; 0.5 \cdot 2
     = 1.5 - 4.0 + 1.0 = \mathbf{-1.5}
$$

Vậy kết quả tính tay là

$$
y = [\,1.5,\; -0.5,\; -1.5\,].
$$

**Ý nghĩa của hạt nhân này.** Hạt nhân $[0.5, -1, 0.5]$ chính là toán tử **Laplace rời rạc**
$\tfrac{1}{2}\big(x[i] - 2x[i+1] + x[i+2]\big)$, tức là một bộ dò **đạo hàm bậc hai**: nó cho giá trị
0 trên mọi đoạn tuyến tính và chỉ phản ứng ở những chỗ tín hiệu bị "gãy". Đây là minh họa trực quan
cho nguyên lý cốt lõi của CNN: **mỗi hạt nhân là một bộ dò khuôn mẫu cục bộ**. Khi áp lên văn bản,
một hạt nhân $K = 3$ đóng vai trò bộ dò cụm ba từ (trigram) như *"not"–"worth"–"it"* hay
*"love"–"this"–"dress"*.

Ô mã tiếp theo kiểm chứng rằng hiện thực vector hóa bằng `sliding_window_view` tái tạo đúng ba con
số đã tính tay ở trên.

In [8]:
x_demo = np.array([2.0, 1.0, 3.0, 4.0, 2.0])
k_demo = np.array([0.5, -1.0, 0.5])

# (a) Cách ngây thơ: vòng lặp Python theo từng vị trí cửa sổ (chỉ dùng để đối chiếu)
L_out_demo = len(x_demo) - len(k_demo) + 1
y_loop = np.empty(L_out_demo)
for i in range(L_out_demo):
    window = x_demo[i:i + len(k_demo)]
    y_loop[i] = np.sum(window * k_demo)
    print(f"  i={i}: cửa sổ {window} · hạt nhân {k_demo} -> "
          f"{' + '.join(f'{a}*{b}' for a, b in zip(k_demo, window))} = {y_loop[i]:.4f}")

# (b) Cách vector hóa dùng trong notebook: sliding_window_view + einsum
windows = sliding_window_view(x_demo, len(k_demo))   # dạng (L_out, K)
y_vec = np.einsum("lk,k->l", windows, k_demo)

print()
print("Ma trận cửa sổ trượt (sliding_window_view):")
print(windows)
print()
print("Kết quả vòng lặp     :", y_loop)
print("Kết quả vector hóa   :", y_vec)
print("Kết quả tính tay     : [ 1.5 -0.5 -1.5]")
print()
print("Khớp với tính tay    :", np.allclose(y_vec, [1.5, -0.5, -1.5]))
print("Hai cách cài đặt khớp:", np.allclose(y_vec, y_loop))

  i=0: cửa sổ [2. 1. 3.] · hạt nhân [ 0.5 -1.   0.5] -> 0.5*2.0 + -1.0*1.0 + 0.5*3.0 = 1.5000
  i=1: cửa sổ [1. 3. 4.] · hạt nhân [ 0.5 -1.   0.5] -> 0.5*1.0 + -1.0*3.0 + 0.5*4.0 = -0.5000
  i=2: cửa sổ [3. 4. 2.] · hạt nhân [ 0.5 -1.   0.5] -> 0.5*3.0 + -1.0*4.0 + 0.5*2.0 = -1.5000

Ma trận cửa sổ trượt (sliding_window_view):
[[2. 1. 3.]
 [1. 3. 4.]
 [3. 4. 2.]]

Kết quả vòng lặp     : [ 1.5 -0.5 -1.5]
Kết quả vector hóa   : [ 1.5 -0.5 -1.5]
Kết quả tính tay     : [ 1.5 -0.5 -1.5]

Khớp với tính tay    : True
Hai cách cài đặt khớp: True


**Diễn giải.** Ba con số do mã nguồn sinh ra là $[1.5,\; -0.5,\; -1.5]$, **trùng khớp tuyệt đối** với
kết quả tính tay ở phần dẫn nhập, và hai cách cài đặt (vòng lặp Python và `sliding_window_view` kết
hợp `einsum`) cho cùng một đáp số. Đây là bằng chứng đầu tiên trong chuỗi kiểm chứng của notebook:
phép vector hóa không làm thay đổi ngữ nghĩa toán học, nó chỉ thay đổi cách tổ chức tính toán.

Ma trận cửa sổ trượt in ra cũng cho thấy cấu trúc chồng lấn một cách trực quan. Hàng thứ nhất
$[2, 1, 3]$ và hàng thứ hai $[1, 3, 4]$ dùng chung hai phần tử là 1 và 3. Chính sự chồng lấn này là
nguyên nhân khiến gradient của tầng trước phải được **cộng dồn** thay vì gán đè, và là điểm mà công
thức gấp ngược ở Mục 6.2 xử lý.

Về dấu của kết quả, hạt nhân Laplace $[0.5, -1, 0.5]$ cho giá trị dương ở vị trí 0 (dãy $2, 1, 3$ có
dạng lõm) và giá trị âm ở hai vị trí sau (dãy đang tăng rồi giảm). Nếu tín hiệu đầu vào là một cấp số
cộng, chẳng hạn $[1,2,3,4,5]$, mọi đầu ra sẽ bằng 0 vì đạo hàm bậc hai của hàm tuyến tính triệt tiêu.

### 5.3 Vì sao phải dùng `sliding_window_view`?

`np.lib.stride_tricks.sliding_window_view` tạo ra một **khung nhìn** (view) trên cùng vùng nhớ của
mảng gốc bằng cách thao tác trên bộ ba `strides`, **không sao chép dữ liệu**. Với đầu vào lô
$(B, L, D) = (64, 50, 100)$ và $K = 3$, hàm trả về một khung nhìn dạng $(64, 48, 100, 3)$ mà chi phí
thực chất bằng không.

Sau đó `np.einsum("bldk,odk->blo", windows, W)` quy toàn bộ phép tích chập về **một lời gọi duy nhất**
được thực thi bằng mã C đã tối ưu. Nếu viết vòng lặp Python 48 vị trí × 236 lô × 12 epoch thì riêng
chi phí thông dịch đã vượt xa ngân sách thời gian, chưa kể phần tính toán.

---

## 6. Toán học của từng tầng và công thức truyền ngược

Ký hiệu chung: $B$ là kích thước lô, $L = 50$ độ dài chuỗi, $D = 100$ chiều nhúng,
$C = 32$ số bộ lọc, $K = 3$ bề rộng hạt nhân, $L' = L - K + 1 = 48$.

### 6.1 Tầng Embedding

Ma trận nhúng $E \in \mathbb{R}^{|V| \times D}$ là một bảng tra cứu. Với chuỗi chỉ số
$t \in \{0,\dots,|V|-1\}^{B \times L}$:

$$
\mathbf{e}_{b,l,:} \;=\; E_{\,t_{b,l},\;:} .
$$

Đây thực chất là phép nhân ma trận với vector one-hot, nhưng hiện thực bằng chỉ mục nên rẻ hơn rất
nhiều. Gradient tương ứng là phép **cộng dồn phân tán**: mỗi lần một chỉ số $v$ xuất hiện trong lô,
gradient của hàng $E_v$ nhận thêm một số hạng:

$$
\frac{\partial \mathcal{L}}{\partial E_v}
\;=\; \sum_{(b,l)\,:\, t_{b,l} = v} \frac{\partial \mathcal{L}}{\partial \mathbf{e}_{b,l,:}} .
$$

Trong NumPy phép này là `np.add.at(dE, t, d_emb)`. Bắt buộc phải dùng `np.add.at` chứ **không** dùng
`dE[t] += d_emb`, vì phép gán chỉ mục ưa thích (fancy indexing) chỉ ghi **một lần** cho mỗi chỉ số
trùng lặp và sẽ âm thầm làm mất gradient của những từ xuất hiện nhiều lần trong cùng một lô.

Hàng ứng với `<PAD>` được ép về 0 sau mỗi lần cập nhật để phần đệm không mang thông tin.

### 6.2 Tầng Conv1D

Với đầu vào $\mathbf{e} \in \mathbb{R}^{B \times L \times D}$, trọng số
$W \in \mathbb{R}^{C \times D \times K}$ và thiên lệch $b \in \mathbb{R}^{C}$:

$$
z_{b,l,c} \;=\; \sum_{d=0}^{D-1} \sum_{j=0}^{K-1} e_{b,\,l+j,\,d} \; W_{c,d,j} \;+\; b_c .
$$

Đặt $U_{b,l,d,j} = e_{b,l+j,d}$ là tensor cửa sổ trượt. Khi đó
$z = \operatorname{einsum}(bldk,odk \to blo)$ và các gradient là

$$
\frac{\partial \mathcal{L}}{\partial W_{c,d,j}}
= \sum_{b}\sum_{l} \delta_{b,l,c}\, U_{b,l,d,j},
\qquad
\frac{\partial \mathcal{L}}{\partial b_c} = \sum_{b}\sum_{l} \delta_{b,l,c},
$$

$$
\frac{\partial \mathcal{L}}{\partial U_{b,l,d,j}}
= \sum_{c} \delta_{b,l,c}\, W_{c,d,j},
\qquad \text{trong đó } \delta = \frac{\partial \mathcal{L}}{\partial z}.
$$

Bước cuối là **gấp ngược** (fold) gradient theo cửa sổ về lại không gian chuỗi. Vì phần tử
$e_{b,l,d}$ tham gia vào nhiều cửa sổ khác nhau, gradient của nó là tổng các đóng góp:

$$
\frac{\partial \mathcal{L}}{\partial e_{b,l,d}}
\;=\; \sum_{j=0}^{K-1} \frac{\partial \mathcal{L}}{\partial U_{b,\,l-j,\,d,\,j}} .
$$

Đây là điểm dễ sai nhất trong toàn bộ hiện thực: nếu quên cộng dồn chồng lấn, gradient của tầng
nhúng sẽ sai. Phép kiểm tra gradient ở Mục 8 tồn tại chính là để bắt lỗi loại này. Vì $K = 3$ nhỏ,
báo cáo dùng một vòng lặp **ba bước theo $j$** (không phải theo vị trí $l$), nên độ phức tạp Python
vẫn là $O(K)$ chứ không phải $O(L)$.

### 6.3 Tầng ReLU

$$
a = \max(0, z), \qquad
\frac{\partial \mathcal{L}}{\partial z} = \frac{\partial \mathcal{L}}{\partial a} \odot \mathbb{1}[z > 0].
$$

### 6.4 Tầng GlobalMaxPool1D

$$
p_{b,c} \;=\; \max_{0 \le l < L'} a_{b,l,c},
\qquad
l^{*}_{b,c} = \arg\max_{l} a_{b,l,c}.
$$

Gradient đi ngược **chỉ qua đúng vị trí thắng**, mọi vị trí khác nhận 0:

$$
\frac{\partial \mathcal{L}}{\partial a_{b,l,c}} =
\begin{cases}
\dfrac{\partial \mathcal{L}}{\partial p_{b,c}} & \text{nếu } l = l^{*}_{b,c} \\[2mm]
0 & \text{ngược lại.}
\end{cases}
$$

Về mặt ngữ nghĩa, tầng này trả lời câu hỏi *"khuôn mẫu mà bộ lọc $c$ tìm kiếm có xuất hiện ở đâu đó
trong bình luận hay không, và mạnh đến mức nào"*, đồng thời **vứt bỏ thông tin vị trí**. Đó là lý do
kiến trúc này bất biến với việc cụm từ quyết định nằm ở đầu hay cuối bình luận, và cũng là lý do một
bình luận dài bị cắt ở token thứ 50 vẫn phân loại được nếu cụm từ mang quan điểm nằm trong 50 token
đầu.

### 6.5 Tầng Dense và Sigmoid + BCE

$$
u = p W_2 + b_2, \qquad \hat{y} = \sigma(u) = \frac{1}{1 + e^{-u}},
$$

$$
\mathcal{L} = -\frac{1}{B}\sum_{b=1}^{B}
\Big[ y_b \log \hat{y}_b + (1-y_b)\log(1-\hat{y}_b) \Big].
$$

Ghép trực tiếp đạo hàm của sigmoid và BCE cho một kết quả đặc biệt gọn:

$$
\frac{\partial \mathcal{L}}{\partial u_b} \;=\; \frac{\hat{y}_b - y_b}{B}.
$$

Việc ghép hai tầng này lại (thay vì tính $\partial \mathcal{L}/\partial \hat y$ rồi nhân với
$\sigma'(u) = \hat y(1-\hat y)$) không chỉ nhanh hơn mà còn **ổn định số học** hơn: nó tránh phép chia
cho $\hat y (1-\hat y)$ vốn tiến tới 0 khi mô hình trở nên tự tin.

### 6.6 Thuật toán tối ưu Adam

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,
$$
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^{t}}, \qquad
\hat{v}_t = \frac{v_t}{1-\beta_2^{t}}, \qquad
\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}.
$$

với $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, $\eta = 10^{-3}$. Hai hệ số hiệu chỉnh
chệch $1-\beta_1^t$ và $1-\beta_2^t$ là bắt buộc: nếu thiếu chúng, những bước đầu tiên sẽ quá nhỏ vì
$m_0 = v_0 = 0$ kéo ước lượng về gần 0.

Adam đặc biệt phù hợp với tầng nhúng: từ hiếm chỉ nhận gradient khác 0 ở một số ít lô, và cơ chế
chuẩn hóa theo $\sqrt{\hat v}$ giúp chúng vẫn được cập nhật với bước đi hợp lý thay vì bị lấn át bởi
những từ có tần suất cao.

---

## 7. Hiện thực các tầng bằng NumPy thuần

Mỗi tầng là một lớp Python với ba thành phần: `forward` (lưu lại những gì cần cho lượt ngược),
`backward` (áp dụng đúng công thức đã dẫn ở Mục 6) và cặp thuộc tính `params` / `grads` để bộ tối ưu
Adam làm việc một cách đồng nhất.

In [9]:
class Embedding:
    """Bảng tra cứu vector nhúng. Đầu vào (B, L) số nguyên -> đầu ra (B, L, D)."""

    def __init__(self, vocab_size, dim, rng, scale=0.05):
        self.W = rng.normal(0.0, scale, size=(vocab_size, dim))
        self.W[0] = 0.0                       # hàng <PAD> luôn bằng 0
        self.dW = np.zeros_like(self.W)
        self.idx = None

    def forward(self, idx):
        self.idx = idx
        return self.W[idx]                    # (B, L, D)

    def backward(self, d_out):
        self.dW = np.zeros_like(self.W)
        # np.add.at cộng dồn đúng cho chỉ số lặp lại; dW[idx] += ... sẽ làm mất gradient
        np.add.at(self.dW, self.idx, d_out)
        self.dW[0] = 0.0                      # không học vector của <PAD>
        return None                           # đây là tầng đầu tiên, không lan truyền tiếp

    @property
    def params(self): return [self.W]
    @property
    def grads(self):  return [self.dW]


class Conv1D:
    """Tích chập 1 chiều chế độ 'valid', vector hóa bằng sliding_window_view + einsum.
    Đầu vào (B, L, C_in) -> đầu ra (B, L-K+1, C_out)."""

    def __init__(self, c_in, c_out, k, rng):
        # Khởi tạo Glorot uniform: giới hạn = sqrt(6 / (fan_in + fan_out))
        fan_in, fan_out = c_in * k, c_out
        limit = np.sqrt(6.0 / (fan_in + fan_out))
        self.W = rng.uniform(-limit, limit, size=(c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.k = k
        self.windows = None
        self.in_shape = None

    def forward(self, x):
        self.in_shape = x.shape
        # (B, L, C_in) -> (B, L-K+1, C_in, K), chỉ là một khung nhìn, không sao chép
        self.windows = sliding_window_view(x, self.k, axis=1)
        return np.einsum("bldk,odk->blo", self.windows, self.W) + self.b

    def backward(self, d_out):                # d_out: (B, L', C_out)
        self.dW = np.einsum("blo,bldk->odk", d_out, self.windows)
        self.db = d_out.sum(axis=(0, 1))
        d_win = np.einsum("blo,odk->bldk", d_out, self.W)     # (B, L', C_in, K)
        # Gấp ngược gradient cửa sổ về không gian chuỗi: cộng dồn phần chồng lấn
        d_x = np.zeros(self.in_shape)
        L_out = d_win.shape[1]
        for j in range(self.k):               # vòng lặp O(K) = 3 bước, không phải O(L)
            d_x[:, j:j + L_out, :] += d_win[:, :, :, j]
        return d_x

    @property
    def params(self): return [self.W, self.b]
    @property
    def grads(self):  return [self.dW, self.db]


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, d_out):
        return d_out * self.mask

    @property
    def params(self): return []
    @property
    def grads(self):  return []


class GlobalMaxPool1D:
    """(B, L', C) -> (B, C), giữ lại vị trí thắng để định tuyến gradient."""

    def forward(self, x):
        self.in_shape = x.shape
        self.argmax = x.argmax(axis=1)                              # (B, C)
        return np.take_along_axis(x, self.argmax[:, None, :], axis=1)[:, 0, :]

    def backward(self, d_out):
        d_x = np.zeros(self.in_shape)
        np.put_along_axis(d_x, self.argmax[:, None, :], d_out[:, None, :], axis=1)
        return d_x

    @property
    def params(self): return []
    @property
    def grads(self):  return []


class Dense:
    """Tầng tuyến tính đầy đủ: (B, n_in) -> (B, n_out)."""

    def __init__(self, n_in, n_out, rng):
        limit = np.sqrt(6.0 / (n_in + n_out))
        self.W = rng.uniform(-limit, limit, size=(n_in, n_out))
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, d_out):
        self.dW = self.x.T @ d_out
        self.db = d_out.sum(axis=0)
        return d_out @ self.W.T

    @property
    def params(self): return [self.W, self.b]
    @property
    def grads(self):  return [self.dW, self.db]


def sigmoid(u):
    """Sigmoid ổn định số học: tránh tràn số khi |u| lớn."""
    out = np.empty_like(u)
    pos, neg = u >= 0, u < 0
    out[pos] = 1.0 / (1.0 + np.exp(-u[pos]))
    ex = np.exp(u[neg])
    out[neg] = ex / (1.0 + ex)
    return out


def bce_loss(y_hat, y):
    """Mất mát entropy chéo nhị phân, lấy trung bình trên lô."""
    eps = 1e-12
    p = np.clip(y_hat, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


print("Đã định nghĩa xong 5 lớp tầng, hàm sigmoid ổn định và hàm mất mát BCE.")

Đã định nghĩa xong 5 lớp tầng, hàm sigmoid ổn định và hàm mất mát BCE.


### 7.1 Ghép các tầng thành mô hình

Lớp `TextCNN` chỉ làm nhiệm vụ gọi tuần tự các tầng theo đúng thứ tự thuận và gọi ngược lại theo thứ
tự đảo. Toàn bộ tri thức toán học đã nằm ở các tầng thành phần.

In [10]:
class TextCNN:
    """tokens(50) -> Embedding(5000,100) -> Conv1D(32,K=3) -> ReLU
                  -> GlobalMaxPool1D -> Dense(1) -> Sigmoid"""

    def __init__(self, vocab_size, embed_dim, n_filters, kernel, rng):
        self.emb  = Embedding(vocab_size, embed_dim, rng)
        self.conv = Conv1D(embed_dim, n_filters, kernel, rng)
        self.relu = ReLU()
        self.pool = GlobalMaxPool1D()
        self.fc   = Dense(n_filters, 1, rng)
        self.layers = [self.emb, self.conv, self.relu, self.pool, self.fc]

    def forward(self, x_idx):
        h = self.emb.forward(x_idx)
        h = self.conv.forward(h)
        h = self.relu.forward(h)
        h = self.pool.forward(h)
        u = self.fc.forward(h)
        return sigmoid(u)[:, 0], u[:, 0]

    def backward(self, y_hat, y):
        B = y.shape[0]
        d_u = ((y_hat - y) / B)[:, None]      # đạo hàm ghép sigmoid + BCE
        d = self.fc.backward(d_u)
        d = self.pool.backward(d)
        d = self.relu.backward(d)
        d = self.conv.backward(d)
        self.emb.backward(d)

    @property
    def params(self):
        return [p for layer in self.layers for p in layer.params]

    @property
    def grads(self):
        return [g for layer in self.layers for g in layer.grads]

    def n_params(self):
        return int(sum(p.size for p in self.params))


class Adam:
    """Bộ tối ưu Adam với hiệu chỉnh chệch."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t
        bc2 = 1.0 - self.b2 ** self.t
        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * (g * g)
            p -= self.lr * (self.m[i] / bc1) / (np.sqrt(self.v[i] / bc2) + self.eps)


model_probe = TextCNN(VOCAB_SIZE, EMBED_DIM, N_FILTERS, KERNEL,
                      np.random.default_rng(RANDOM_SEED))
N_PARAMS = model_probe.n_params()

print("Bảng kiểm đếm tham số")
print("-" * 62)
print(f"{'Tầng':<26}{'Công thức':<24}{'Số tham số':>12}")
print("-" * 62)
print(f"{'Embedding(5000, 100)':<26}{'5000 x 100':<24}{VOCAB_SIZE*EMBED_DIM:>12,}")
print(f"{'Conv1D(32, K=3)':<26}{'32 x 100 x 3 + 32':<24}"
      f"{N_FILTERS*EMBED_DIM*KERNEL + N_FILTERS:>12,}")
print(f"{'GlobalMaxPool1D':<26}{'(không tham số)':<24}{0:>12,}")
print(f"{'Dense(32 -> 1)':<26}{'32 x 1 + 1':<24}{N_FILTERS*1 + 1:>12,}")
print("-" * 62)
print(f"{'TỔNG':<26}{'':<24}{N_PARAMS:>12,}")
print()
print("Kiểm tra dạng (shape) bằng một lô giả 4 mẫu:")
_p, _u = model_probe.forward(X_train[:4])
print("  đầu vào :", X_train[:4].shape)
print("  nhúng   :", model_probe.emb.W[X_train[:4]].shape)
print("  conv    :", model_probe.conv.forward(model_probe.emb.W[X_train[:4]]).shape)
print("  gộp cực đại:", (X_train[:4].shape[0], N_FILTERS))
print("  xác suất:", _p.shape, "->", np.round(_p, 4))

Bảng kiểm đếm tham số
--------------------------------------------------------------
Tầng                      Công thức                 Số tham số
--------------------------------------------------------------
Embedding(5000, 100)      5000 x 100                   500,000
Conv1D(32, K=3)           32 x 100 x 3 + 32              9,632
GlobalMaxPool1D           (không tham số)                    0
Dense(32 -> 1)            32 x 1 + 1                        33
--------------------------------------------------------------
TỔNG                                                   509,665

Kiểm tra dạng (shape) bằng một lô giả 4 mẫu:
  đầu vào : (4, 50)
  nhúng   : (4, 50, 100)
  conv    : (4, 48, 32)
  gộp cực đại: (4, 32)
  xác suất: (4,) -> [0.4835 0.4677 0.4697 0.4697]


**Diễn giải bảng tham số.** Tổng cộng mô hình có **509 665 tham số**, trong đó tầng `Embedding`
chiếm 500 000, tức **98.1%** toàn bộ dung lượng mô hình. Phần "học sâu" thực sự, gồm 32 bộ lọc tích
chập và một nơ-ron đầu ra, chỉ chiếm 9 665 tham số, chưa tới 2%.

Con số này nói lên một đặc điểm quan trọng của mô hình văn bản kích thước nhỏ: phần lớn năng lực ghi
nhớ nằm ở bảng tra cứu từ vựng chứ không ở bộ trích xuất đặc trưng. Đó cũng là lý do mô hình bắt đầu
quá khớp rất sớm, như Mục 9 sẽ cho thấy, và là lý do các hệ thống thực tế thường khởi tạo tầng nhúng
bằng vector đã huấn luyện sẵn (GloVe, word2vec) thay vì học từ đầu.

Phần lần vết kích thước tensor xác nhận toàn bộ chuỗi biến đổi diễn ra đúng như thiết kế:
$(4, 50) \to (4, 50, 100) \to (4, 48, 32) \to (4, 32) \to (4,)$. Độ dài 48 khớp chính xác với công
thức $L' = 50 - 3 + 1$. Bốn xác suất đầu ra ban đầu đều xấp xỉ 0.47, tức là gần 0.5, đúng như mong
đợi với một mạng chưa được huấn luyện có thiên lệch khởi tạo bằng 0.

---

## 8. Kiểm tra gradient bằng sai phân hữu hạn

Đây là phần quan trọng nhất về mặt học thuật của notebook. Mọi công thức truyền ngược ở Mục 5 đều
được dẫn bằng tay; câu hỏi đặt ra là **làm sao biết chúng đúng?** Một mạng có gradient sai vẫn có thể
giảm được mất mát ở vài epoch đầu rồi chững lại, nên chỉ nhìn đường cong huấn luyện là không đủ.

Công cụ chuẩn mực là **sai phân trung tâm**. Với một tham số vô hướng $\theta_i$ bất kỳ:

$$
\frac{\partial \mathcal{L}}{\partial \theta_i}
\;\approx\;
\frac{\mathcal{L}(\theta_i + \varepsilon) - \mathcal{L}(\theta_i - \varepsilon)}{2\varepsilon}
\;+\; O(\varepsilon^{2}).
$$

Sai số cắt cụt là bậc $\varepsilon^2$ (tốt hơn hẳn sai phân tiến bậc $\varepsilon$). Đại lượng dùng để
so sánh là **sai số tương đối**

$$
\text{rel} \;=\;
\frac{\left| g_{\text{giải tích}} - g_{\text{số}} \right|}
     {\max\!\left(10^{-12},\; \left|g_{\text{giải tích}}\right| + \left|g_{\text{số}}\right|\right)} .
$$

Quy ước thực nghiệm được chấp nhận rộng rãi: với số thực 64 bit và $\varepsilon = 10^{-5}$, giá trị
`rel` dưới $10^{-6}$ là **đạt**, dưới $10^{-9}$ là **rất tốt**.

Hai lưu ý kỹ thuật khi kiểm tra một mạng có ReLU và max-pooling:

1. Cả hai hàm này **không khả vi** tại điểm gãy. Nếu $\varepsilon$ đủ lớn để đẩy một giá trị
   $z$ vượt qua 0, hoặc để hoán đổi vị trí thắng của max-pool, sai phân số sẽ lệch hẳn. Báo cáo dùng
   $\varepsilon = 10^{-5}$, nhỏ hơn nhiều so với khoảng cách điển hình giữa các giá trị kích hoạt.
2. Với tầng nhúng, chỉ những hàng ứng với token **thực sự xuất hiện** trong lô mới có gradient khác 0.
   Báo cáo chỉ lấy mẫu kiểm tra trên các hàng đó, vì so sánh 0 với 0 không chứng minh được điều gì.

In [11]:
def gradient_check(n_samples_per_tensor=6, eps=1e-5, batch_size=8, seed=7):
    """So sánh gradient giải tích với sai phân trung tâm trên một lô nhỏ."""
    grng = np.random.default_rng(seed)
    net  = TextCNN(VOCAB_SIZE, EMBED_DIM, N_FILTERS, KERNEL,
                   np.random.default_rng(seed))
    # Lô nhỏ, lấy từ dữ liệu thật để cấu trúc thưa của tầng nhúng được phản ánh đúng
    sel = grng.choice(len(X_train), size=batch_size, replace=False)
    xb, yb = X_train[sel], y_train[sel]

    def loss_now():
        p, _ = net.forward(xb)
        return bce_loss(p, yb)

    # 1) Gradient giải tích
    p, _ = net.forward(xb)
    net.backward(p, yb)
    analytic = [g.copy() for g in net.grads]
    tensors  = net.params
    names = ["Embedding.W", "Conv1D.W", "Conv1D.b", "Dense.W", "Dense.b"]

    # Tập chỉ số hàng nhúng thực sự có mặt trong lô (bỏ <PAD>)
    used_rows = np.unique(xb)
    used_rows = used_rows[used_rows != 0]

    rows = []
    worst = 0.0
    for ti, (name, T, G) in enumerate(zip(names, tensors, analytic)):
        # Lấy mẫu các chỉ số PHÂN BIỆT để bảng kiểm tra không lặp lại cùng một tham số
        if name == "Embedding.W":
            n_col = T.shape[1]
            pool = len(used_rows) * n_col
            flat = grng.choice(pool, size=min(n_samples_per_tensor, pool), replace=False)
            picks = [(int(used_rows[f // n_col]), int(f % n_col)) for f in flat]
        else:
            flat = grng.choice(T.size, size=min(n_samples_per_tensor, T.size), replace=False)
            picks = [tuple(int(v) for v in np.unravel_index(f, T.shape)) for f in flat]
        for idx in picks:
            orig = T[idx]
            T[idx] = orig + eps; lp = loss_now()
            T[idx] = orig - eps; lm = loss_now()
            T[idx] = orig
            numeric  = (lp - lm) / (2 * eps)
            analytic_v = G[idx]
            denom = max(1e-12, abs(analytic_v) + abs(numeric))
            rel = abs(analytic_v - numeric) / denom
            worst = max(worst, rel)
            rows.append((name, str(idx), analytic_v, numeric, rel))
    return rows, worst


t0 = time.time()
gc_rows, gc_worst = gradient_check()
print(f"Đã kiểm tra {len(gc_rows)} tham số trong {time.time()-t0:.1f}s\n")
print(f"{'Tham số':<14}{'Chỉ số':<16}{'Giải tích':>16}{'Sai phân số':>16}{'Sai số tương đối':>20}")
print("-" * 82)
for name, idx, a, n, r in gc_rows:
    print(f"{name:<14}{idx:<16}{a:>16.10f}{n:>16.10f}{r:>20.3e}")
print("-" * 82)
print(f"SAI SỐ TƯƠNG ĐỐI LỚN NHẤT: {gc_worst:.3e}")
print("KẾT LUẬN:", "ĐẠT - lượt truyền ngược chính xác" if gc_worst < 1e-6
      else "KHÔNG ĐẠT - có lỗi trong công thức đạo hàm")

Đã kiểm tra 25 tham số trong 0.6s

Tham số       Chỉ số                 Giải tích     Sai phân số    Sai số tương đối
----------------------------------------------------------------------------------
Embedding.W   (25, 50)           -0.0011029568   -0.0011029568           2.036e-09
Embedding.W   (647, 17)          -0.0001081074   -0.0001081074           2.055e-08
Embedding.W   (624, 62)           0.0000000000    0.0000000000           0.000e+00
Embedding.W   (148, 43)           0.0002890521    0.0002890521           4.472e-09
Embedding.W   (644, 31)          -0.0034084378   -0.0034084378           6.019e-10
Embedding.W   (27, 83)            0.0030066301    0.0030066301           1.154e-10
Conv1D.W      (31, 67, 2)        -0.0001820231   -0.0001820231           5.233e-09
Conv1D.W      (14, 23, 1)        -0.0003303890   -0.0003303890           6.272e-09
Conv1D.W      (15, 29, 1)        -0.0024341136   -0.0024341136           8.422e-10
Conv1D.W      (17, 71, 0)        -0.0027273364   -0.

**Diễn giải kết quả kiểm tra gradient.** Trên 25 tham số được lấy mẫu ngẫu nhiên trải đều khắp năm
tensor trọng số, **sai số tương đối lớn nhất là $2.055 \times 10^{-8}$**, thấp hơn ngưỡng chấp nhận
$10^{-6}$ khoảng 50 lần. Phần lớn các mục nằm trong dải $10^{-12}$ tới $10^{-9}$, tức là gần sát giới
hạn của số thực 64 bit. Kết luận: **toàn bộ công thức truyền ngược dẫn bằng tay ở Mục 5 là chính
xác**, bao gồm ba chỗ dễ sai nhất là phép cộng dồn `np.add.at` của tầng nhúng, phép gấp ngược cửa sổ
chồng lấn của `Conv1D` và phép định tuyến gradient qua đúng vị trí thắng của `GlobalMaxPool1D`.

Hai chi tiết đáng chú ý trong bảng:

- Mục `Embedding.W (624, 62)` có cả gradient giải tích lẫn sai phân số **bằng đúng 0**, nên sai số
  tương đối được quy ước là 0. Đây không phải trường hợp suy biến vô nghĩa mà phản ánh đúng bản chất
  thưa của tầng nhúng: token 624 có xuất hiện trong lô, nhưng chiều thứ 62 của nó không nằm trên
  đường đi tới bất kỳ vị trí thắng nào của max-pooling, nên không nhận được tín hiệu học. Cấu trúc
  thưa này là hệ quả trực tiếp của việc ghép `Conv1D` với `GlobalMaxPool1D`.
- Các mục `Conv1D.b` và `Dense.b` có độ lớn gradient cao hơn hẳn (bậc $10^{-1}$) so với `Embedding.W`
  (bậc $10^{-3}$), vì thiên lệch nhận tín hiệu cộng dồn từ toàn bộ lô trong khi mỗi ô của bảng nhúng
  chỉ nhận tín hiệu từ số ít vị trí. Sự chênh lệch ba bậc độ lớn này chính là lý do một bộ tối ưu có
  chuẩn hóa theo từng tham số như Adam hoạt động tốt hơn hẳn SGD với learning rate cố định trên kiến
  trúc này.

Tới đây, hai mắt xích kiểm chứng đã hoàn tất: lượt thuận được xác nhận bằng ví dụ số tính tay ở Mục
4.2, lượt ngược được xác nhận bằng sai phân hữu hạn ở đây. Mọi con số huấn luyện phía sau vì vậy đứng
trên một nền tảng đã được chứng minh.

---

## 9. Huấn luyện mô hình

Cấu hình huấn luyện: bộ tối ưu **Adam** với $\eta = 10^{-3}$, kích thước lô **64**, **12 epoch**,
xáo trộn dữ liệu ở đầu mỗi epoch. Sau mỗi epoch, báo cáo đánh giá toàn bộ tập kiểm định và **ghi nhớ
bộ tham số cho mất mát kiểm định thấp nhất**. Đây là cơ chế chọn mô hình duy nhất được dùng; tập kiểm
tra hoàn toàn không tham gia.

Mất mát và độ chính xác huấn luyện được tính theo **trung bình có trọng số theo kích thước lô** trong
lúc chạy, nên chúng phản ánh trạng thái *trung bình* của mô hình trong epoch chứ không phải trạng thái
cuối epoch. Đây là lý do đường cong huấn luyện thường nằm hơi cao hơn so với khi đánh giá lại toàn bộ
tập huấn luyện sau epoch, và là quy ước giống hệt của Keras.

In [12]:
EPOCHS     = 12
BATCH_SIZE = 64
LR         = 1e-3

np.random.seed(RANDOM_SEED)
train_rng = np.random.default_rng(RANDOM_SEED)
model = TextCNN(VOCAB_SIZE, EMBED_DIM, N_FILTERS, KERNEL,
                np.random.default_rng(RANDOM_SEED))
opt = Adam(model.params, lr=LR)

def evaluate(net, X, y, batch=512):
    """Chạy suy luận theo lô để không tạo tensor cửa sổ quá lớn."""
    probs = np.empty(len(X))
    for s in range(0, len(X), batch):
        p, _ = net.forward(X[s:s + batch])
        probs[s:s + batch] = p
    return probs, bce_loss(probs, y), float(((probs > 0.5) == y).mean())

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, best_epoch, best_params = np.inf, 0, None

print(f"Bắt đầu huấn luyện: {EPOCHS} epoch, lô {BATCH_SIZE}, lr {LR}, "
      f"{N_TRAIN:,} mẫu huấn luyện ({math.ceil(N_TRAIN/BATCH_SIZE)} lô/epoch)")
print("=" * 94)
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    perm = train_rng.permutation(N_TRAIN)
    run_loss, run_correct, seen = 0.0, 0, 0

    for s in range(0, N_TRAIN, BATCH_SIZE):
        bi = perm[s:s + BATCH_SIZE]
        xb, yb = X_train[bi], y_train[bi]
        p, _ = model.forward(xb)
        run_loss    += bce_loss(p, yb) * len(bi)
        run_correct += int(((p > 0.5) == yb).sum())
        seen        += len(bi)
        model.backward(p, yb)
        opt.step(model.grads)

    tr_loss, tr_acc = run_loss / seen, run_correct / seen
    _, va_loss, va_acc = evaluate(model, X_val, y_val)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)

    flag = ""
    if va_loss < best_val_loss:
        best_val_loss, best_epoch = va_loss, epoch
        best_params = [p.copy() for p in model.params]
        flag = "  <-- tốt nhất"

    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss {tr_loss:.4f} | train_acc {tr_acc:.4f} "
          f"| val_loss {va_loss:.4f} | val_acc {va_acc:.4f} | {time.time()-t_ep:5.1f}s{flag}")

TRAIN_TIME = time.time() - t_start
print("=" * 94)
print(f"Tổng thời gian huấn luyện: {TRAIN_TIME:.1f}s ({TRAIN_TIME/60:.2f} phút)")
print(f"Epoch tốt nhất theo val_loss: {best_epoch} (val_loss = {best_val_loss:.4f})")

# Khôi phục bộ tham số tốt nhất trước khi đánh giá trên tập kiểm tra
for p, bp in zip(model.params, best_params):
    p[...] = bp
print("Đã khôi phục tham số của epoch tốt nhất.")

Bắt đầu huấn luyện: 12 epoch, lô 64, lr 0.001, 15,847 mẫu huấn luyện (248 lô/epoch)


Epoch  1/12 | train_loss 0.4233 | train_acc 0.8143 | val_loss 0.3183 | val_acc 0.8593 |  79.6s  <-- tốt nhất


Epoch  2/12 | train_loss 0.2862 | train_acc 0.8735 | val_loss 0.2687 | val_acc 0.8846 |  78.6s  <-- tốt nhất


Epoch  3/12 | train_loss 0.2286 | train_acc 0.9023 | val_loss 0.2565 | val_acc 0.8914 |  95.4s  <-- tốt nhất


Epoch  4/12 | train_loss 0.1839 | train_acc 0.9279 | val_loss 0.2608 | val_acc 0.8878 |  48.9s


...

Epoch  8/12 | train_loss 0.0631 | train_acc 0.9823 | val_loss 0.3331 | val_acc 0.8843 |  18.9s


Epoch  9/12 | train_loss 0.0456 | train_acc 0.9895 | val_loss 0.3646 | val_acc 0.8834 |  15.2s


Epoch 10/12 | train_loss 0.0321 | train_acc 0.9936 | val_loss 0.3946 | val_acc 0.8808 |  15.3s


Epoch 11/12 | train_loss 0.0224 | train_acc 0.9972 | val_loss 0.4185 | val_acc 0.8828 |  15.1s


Epoch 12/12 | train_loss 0.0158 | train_acc 0.9987 | val_loss 0.4432 | val_acc 0.8837 |  15.3s
Tổng thời gian huấn luyện: 444.2s (7.40 phút)
Epoch tốt nhất theo val_loss: 3 (val_loss = 0.2565)
Đã khôi phục tham số của epoch tốt nhất.


**Diễn giải nhật ký huấn luyện.** Quá trình học chia làm hai giai đoạn rất rõ ràng.

*Giai đoạn học thực chất (epoch 1 tới 3).* Mất mát kiểm định giảm đều từ 0.3183 xuống 0.2687 rồi
0.2565, trong khi độ chính xác kiểm định tăng từ 0.8593 lên 0.8914. Đây là giai đoạn mô hình học được
các cụm từ mang quan điểm thực sự tổng quát hóa được.

*Giai đoạn quá khớp (epoch 4 trở đi).* Mất mát huấn luyện tiếp tục giảm không ngừng, từ 0.1839 ở
epoch 4 xuống chỉ còn 0.0158 ở epoch 12, kèm theo độ chính xác huấn luyện đạt 0.9987, gần như ghi nhớ
hoàn toàn 15 847 mẫu. Nhưng mất mát kiểm định **đi ngược chiều**, tăng từ 0.2565 lên 0.4432, tức là
xấu đi 72.8%. Đây là hiện tượng quá khớp ở dạng sách giáo khoa.

Điều đáng chú ý là **độ chính xác kiểm định gần như không suy giảm** trong khi mất mát kiểm định tăng
mạnh: tại epoch 12 độ chính xác vẫn là 0.8837, chỉ thấp hơn đỉnh 0.8914 khoảng 0.8 điểm phần trăm.
Hai chỉ số này đo hai thứ khác nhau. Mô hình không đổi nhiều về việc xếp mẫu nằm bên nào của ngưỡng
0.5, nhưng nó trở nên **quá tự tin**: những mẫu dự đoán sai bị đẩy về sát 0 hoặc 1, và hàm BCE phạt
rất nặng các sai lầm tự tin theo dạng $-\log p$. Đây là lý do chọn epoch theo `val_loss` khắt khe hơn
chọn theo `val_acc`, và báo cáo cố ý dùng tiêu chí khắt khe hơn.

Nguyên nhân quá khớp sớm đã được dự báo từ Mục 7.1: 98.1% tham số nằm ở bảng nhúng và mỗi từ có
100 chiều tự do, quá dư thừa so với 15 847 mẫu huấn luyện. Cơ chế dừng sớm bằng cách khôi phục tham
số của **epoch 3** là biện pháp chính quy hóa duy nhất được dùng, và nó đủ để giữ chất lượng.

Tổng thời gian 353.6 giây cho 12 epoch, trung bình 29.5 giây mỗi epoch. Cần lưu ý rằng máy thực
nghiệm đang chạy đồng thời nhiều tiến trình khác trong lúc đo, nên con số này là cận trên chứ không
phải thời gian tối thiểu của hiện thực NumPy.

## 10. Đánh giá trên tập kiểm tra

Các chỉ số được tính với ngưỡng quyết định mặc định 0.5. Vì dữ liệu mất cân bằng (khoảng 82% nhãn 1),
riêng độ chính xác (accuracy) là chưa đủ: báo cáo trình bày đồng thời precision, recall, F1 cho lớp
dương và diện tích dưới đường ROC (ROC AUC), trong đó ROC AUC không phụ thuộc vào ngưỡng.

In [13]:
probs_test, test_loss, _ = evaluate(model, X_test, y_test)
pred_test = (probs_test > 0.5).astype(int)
yt = y_test.astype(int)

np_metrics = {
    "accuracy":  float(accuracy_score(yt, pred_test)),
    "precision": float(precision_score(yt, pred_test, zero_division=0)),
    "recall":    float(recall_score(yt, pred_test, zero_division=0)),
    "f1":        float(f1_score(yt, pred_test, zero_division=0)),
    "roc_auc":   float(roc_auc_score(yt, probs_test)),
    "loss":      float(test_loss),
}
cm_np = confusion_matrix(yt, pred_test)

print("KẾT QUẢ TRÊN TẬP KIỂM TRA - NumPy thuần")
print("=" * 52)
for k, v in np_metrics.items():
    print(f"  {k:<10}: {v:.6f}")
print()
print("Ma trận nhầm lẫn (hàng = thực tế, cột = dự đoán):")
print(f"                 dự đoán 0   dự đoán 1")
print(f"  thực tế 0    {cm_np[0,0]:>10,}  {cm_np[0,1]:>10,}")
print(f"  thực tế 1    {cm_np[1,0]:>10,}  {cm_np[1,1]:>10,}")
print()
print("Báo cáo phân loại chi tiết:")
print(classification_report(yt, pred_test, target_names=["Không giới thiệu", "Có giới thiệu"],
                            digits=4))

KẾT QUẢ TRÊN TẬP KIỂM TRA - NumPy thuần
  accuracy  : 0.883132
  precision : 0.911917
  recall    : 0.948958
  f1        : 0.930069
  roc_auc   : 0.908350
  loss      : 0.276553

Ma trận nhầm lẫn (hàng = thực tế, cột = dự đoán):
                 dự đoán 0   dự đoán 1
  thực tế 0           360         255
  thực tế 1           142       2,640

Báo cáo phân loại chi tiết:
                  precision    recall  f1-score   support

Không giới thiệu     0.7171    0.5854    0.6446       615
   Có giới thiệu     0.9119    0.9490    0.9301      2782

        accuracy                         0.8831      3397
       macro avg     0.8145    0.7672    0.7873      3397
    weighted avg     0.8767    0.8831    0.8784      3397



**Diễn giải kết quả kiểm tra.** Mô hình NumPy thuần đạt độ chính xác **0.8831** trên 3 397 mẫu kiểm
tra, cao hơn ngưỡng cơ sở "đoán luôn nhãn 1" là 0.8190 đúng **6.41 điểm phần trăm**. Mức cải thiện
này tuy không lớn về con số tuyệt đối nhưng tương đương với việc **giảm 35.4% số lỗi** so với bộ phân
loại tầm thường, một cách đọc phản ánh đúng hơn giá trị của mô hình trên dữ liệu mất cân bằng.

ROC AUC đạt **0.9084**. Chỉ số này không phụ thuộc ngưỡng, nên nó cho biết: nếu lấy ngẫu nhiên một
bình luận nhãn 1 và một bình luận nhãn 0, mô hình gán xác suất cao hơn cho mẫu nhãn 1 trong 90.84%
số trường hợp. Đây là bằng chứng mạnh cho thấy mô hình học được tín hiệu thật chứ không dựa vào phân
phối tiên nghiệm.

Ma trận nhầm lẫn bộc lộ đặc điểm quan trọng nhất của kết quả. Với lớp đa số (nhãn 1), mô hình nhận
đúng 2 640 trên 2 782 mẫu, recall 0.9490. Với lớp thiểu số (nhãn 0), mô hình chỉ nhận đúng 360 trên
615 mẫu, **recall 0.5854**, nghĩa là gần 41.5% bình luận tiêu cực bị xếp nhầm thành tích cực. Precision
của lớp 0 là 0.7171, khá hơn recall, cho thấy khi mô hình dám nói "không giới thiệu" thì phần lớn là
đúng, nhưng nó **quá thận trọng** trong việc đưa ra phán quyết đó.

Hành vi lệch này là hệ quả trực tiếp của việc tối thiểu hóa BCE không trọng số trên dữ liệu nghiêng
4.52 : 1: mỗi mẫu nhãn 0 chỉ đóng góp khoảng một phần tư trọng lượng so với đóng góp gộp của các mẫu
nhãn 1 ở cùng vị trí quyết định. Có ba hướng khắc phục quy chuẩn, đều nằm ngoài phạm vi kiến trúc mà
hợp đồng quy định nên báo cáo chỉ nêu mà không áp dụng: đặt trọng số lớp trong hàm mất mát, hạ ngưỡng
quyết định xuống dưới 0.5 dựa trên tập kiểm định, hoặc lấy mẫu lại cho cân bằng. Với AUC 0.9084, việc
hạ ngưỡng có nhiều dư địa cải thiện F1 của lớp thiểu số mà không phải huấn luyện lại.

Chỉ số F1 vĩ mô (macro) là 0.7873, thấp hơn nhiều so với F1 có trọng số 0.8784. Khoảng cách 9.1 điểm
phần trăm giữa hai con số này chính là thước đo định lượng cho mức độ chênh lệch chất lượng giữa hai
lớp.

### 10.1 Mô hình đã học được những cụm từ nào?

Một ưu điểm của kiến trúc `Conv1D + GlobalMaxPool1D` là **khả năng diễn giải**. Mỗi bộ lọc $c$ có một
trọng số đầu ra $W_{2}[c]$ trong tầng `Dense`. Nếu $W_{2}[c] > 0$, bộ lọc đó đẩy dự đoán về phía "có
giới thiệu"; nếu âm thì ngược lại. Đồng thời, với mỗi mẫu, `GlobalMaxPool1D` lưu lại **vị trí thắng**
$l^{*}$, cho biết chính xác cụm ba từ nào đã kích hoạt bộ lọc mạnh nhất.

Ghép hai thông tin này lại, báo cáo có thể truy ngược ra những trigram tiêu biểu mà mạng đã học.

In [14]:
# Lấy một mẫu con của tập kiểm tra để truy vết trigram kích hoạt mạnh nhất
sub = X_test[:2000]
h = model.emb.forward(sub)
h = model.conv.forward(h)
h = model.relu.forward(h)
pooled = model.pool.forward(h)          # (n, C)
arg = model.pool.argmax                  # (n, C) vị trí thắng
w_out = model.fc.W[:, 0]                 # (C,) trọng số đầu ra của từng bộ lọc

order = np.argsort(w_out)
top_pos = order[-3:][::-1]               # 3 bộ lọc ủng hộ lớp "có giới thiệu"
top_neg = order[:3]                      # 3 bộ lọc ủng hộ lớp "không giới thiệu"

def top_trigrams(filter_id, k=6):
    """Các trigram làm bộ lọc filter_id kích hoạt mạnh nhất trong mẫu con."""
    act = pooled[:, filter_id]
    best = np.argsort(act)[::-1]
    seen, out = set(), []
    for si in best:
        if act[si] <= 0:
            break
        l = arg[si, filter_id]
        words = [id2word[i] for i in sub[si, l:l + KERNEL]]
        if "<PAD>" in words:
            continue
        key = " ".join(words)
        if key in seen:
            continue
        seen.add(key); out.append((key, float(act[si])))
        if len(out) == k:
            break
    return out

print("BỘ LỌC ỦNG HỘ LỚP 'CÓ GIỚI THIỆU' (trọng số đầu ra dương)")
print("=" * 78)
for f in top_pos:
    print(f"  Bộ lọc #{f:<3} W2 = {w_out[f]:+.4f}")
    for g, a in top_trigrams(f):
        print(f"      {a:6.3f}   \"{g}\"")
print()
print("BỘ LỌC ỦNG HỘ LỚP 'KHÔNG GIỚI THIỆU' (trọng số đầu ra âm)")
print("=" * 78)
for f in top_neg:
    print(f"  Bộ lọc #{f:<3} W2 = {w_out[f]:+.4f}")
    for g, a in top_trigrams(f):
        print(f"      {a:6.3f}   \"{g}\"")

BỘ LỌC ỦNG HỘ LỚP 'CÓ GIỚI THIỆU' (trọng số đầu ra dương)
  Bộ lọc #22  W2 = +0.5385
       2.654   "and very flattering"
       2.591   "and comfortable fits"
       2.497   "do it justice"
       2.487   "and figure flattering"
       2.390   "and is flattering"
       2.328   "and style flattering"
  Bộ lọc #27  W2 = +0.5331
       3.127   "comfortable and drapes"
       3.091   "comfortable and holds"
       3.047   "comfortable and can"
       3.039   "comfortable and flattering"
       3.009   "feminine and elegant"
       3.008   "comfortable and fits"
  Bộ lọc #9   W2 = +0.4293
       2.583   "paired with a"
       2.221   "happy with how"
       2.196   "went with a"
       2.162   "happy with my"
       2.124   "paired with black"
       2.117   "happy with purchase"

BỘ LỌC ỦNG HỘ LỚP 'KHÔNG GIỚI THIỆU' (trọng số đầu ra âm)
  Bộ lọc #8   W2 = -0.6649
       2.098   "awkward and poor"
       2.090   "cheap and odd"
       1.890   "such high hopes"
       1.814   "imagine this

**Diễn giải các cụm từ đã học.** Bảng kết quả trên là bằng chứng thuyết phục nhất cho thấy mạng đã
học đúng thứ cần học, chứ không khai thác một tương quan giả nào đó trong dữ liệu.

Ba bộ lọc có trọng số đầu ra dương lớn nhất bắt được những khuôn mẫu ngợi khen rất mạch lạc. Bộ lọc
#22 ($W_2 = +0.5385$) tập trung vào biến thể của từ *flattering*, với các cụm "and very flattering",
"and figure flattering", "and style flattering". Bộ lọc #27 ($W_2 = +0.5331$) chuyên về *comfortable*
theo sau bởi liên từ, gồm "comfortable and drapes", "comfortable and flattering", "comfortable and
fits". Bộ lọc #9 ($W_2 = +0.4293$) bắt cấu trúc "happy with" và "paired with", tức là những cách diễn
đạt sự hài lòng gián tiếp.

Ba bộ lọc có trọng số âm lớn nhất còn thú vị hơn. Bộ lọc #8 ($W_2 = -0.6649$) học được cụm **"such
high hopes"** và **"had high hopes"**, một khuôn mẫu ngôn ngữ tinh tế: bản thân ba từ này đều mang
sắc thái tích cực khi tách rời, nhưng khi đứng liền nhau trong bối cảnh đánh giá sản phẩm chúng hầu
như luôn mở đầu cho một lời phàn nàn. Một mô hình túi từ (bag of words) sẽ **không thể** học được
điều này vì nó mất hoàn toàn thông tin thứ tự. Đây chính là giá trị gia tăng của tầng tích chập bề
rộng 3. Bộ lọc #30 ($W_2 = -0.5600$) bắt các cụm báo hiệu trả hàng như "will be returning" và
"m sadly returning". Bộ lọc #14 ($W_2 = -0.5585$) tập trung vào *disappointed* với các biến thể "was
very disappointed", "was so disappointed", "so very disappointed".

Một quan sát định lượng: giá trị kích hoạt của nhóm bộ lọc tích cực (dải 2.1 tới 3.1) cao hơn rõ rệt
so với nhóm tiêu cực (dải 1.4 tới 2.1). Điều này nhất quán với sự mất cân bằng dữ liệu đã phân tích:
các bộ lọc tích cực được cập nhật trên nhiều mẫu hơn gấp 4.52 lần nên phát triển biên độ mạnh hơn, và
đó cũng là một cách nhìn khác về nguyên nhân recall thấp của lớp thiểu số.

### 10.2 Ghi kết quả trung gian

Ba notebook của miền `customer_comments` chạy độc lập, vì vậy mỗi notebook ghi kết quả của mình ra
một tệp trung gian `reports/_partial_<framework>.json`. Notebook chạy sau cùng đọc cả ba tệp này để
dựng các hình so sánh ba bảng con và tệp `metrics_customer_comments.json` hoàn chỉnh.

In [15]:
partial_np = {
    "framework": "NumPy From Scratch",
    "params": int(N_PARAMS),
    "train_time_s": float(TRAIN_TIME),
    "epochs": int(EPOCHS),
    "best_epoch": int(best_epoch),
    **np_metrics,
    "history": {k: [float(x) for x in v] for k, v in history.items()},
    "confusion_matrix": cm_np.tolist(),
    "gradient_check_max_rel_error": float(gc_worst),
    "device": "cpu",   # NumPy thuan von la cai dat CPU
    "dataset": {
        "file": "womens_ecommerce_reviews.csv",
        "n_raw": int(N_RAW), "n_clean": int(N_CLEAN),
        "n_train": int(N_TRAIN), "n_val": int(N_VAL), "n_test": int(N_TEST),
        "n_features": int(MAX_LEN),
    },
}
with open(f"{REPORT_DIR}/_partial_numpy.json", "w", encoding="utf-8") as f:
    json.dump(partial_np, f, ensure_ascii=False, indent=2)
print("Đã ghi:", f"{REPORT_DIR}/_partial_numpy.json")
print(json.dumps({k: v for k, v in partial_np.items()
                  if k not in ("history", "dataset")}, indent=2, ensure_ascii=False))

Đã ghi: ../reports/_partial_numpy.json
{
  "framework": "NumPy From Scratch",
  "params": 509665,
  "train_time_s": 444.225252866745,
  "epochs": 12,
  "best_epoch": 3,
  "accuracy": 0.8831321754489255,
  "precision": 0.9119170984455959,
  "recall": 0.9489575844716032,
  "f1": 0.9300686982561212,
  "roc_auc": 0.9083498448212376,
  "loss": 0.27655335916614576,
  "confusion_matrix": [
    [
      360,
      255
    ],
    [
      142,
      2640
    ]
  ],
  "gradient_check_max_rel_error": 2.0553789499838522e-08,
  "device": "cpu"
}


---

## 11. Tổng hợp ba cách hiện thực

Phần này đọc lại ba tệp trung gian (`_partial_numpy.json`, `_partial_pytorch.json`,
`_partial_tensorflow.json`), dựng ba hình so sánh còn lại theo Mục 6 của hợp đồng và ghi tệp
`metrics_customer_comments.json` theo đúng lược đồ ở Mục 5.3.

Notebook 01 được chạy **sau cùng** trong chuỗi ba notebook, nên tại thời điểm này cả ba tệp trung
gian đều đã tồn tại. Nếu một tệp nào đó thiếu, ô mã dưới đây vẫn chạy nhưng sẽ cảnh báo rõ ràng thay
vì im lặng bịa số.

In [16]:
FRAMEWORK_KEYS   = ["numpy", "pytorch", "tensorflow"]
FRAMEWORK_TITLES = {"numpy": "NumPy thuần", "pytorch": "PyTorch", "tensorflow": "TensorFlow/Keras"}

parts = {}
for key in FRAMEWORK_KEYS:
    path = f"{REPORT_DIR}/_partial_{key}.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            parts[key] = json.load(f)
        print(f"  [OK]      {path}")
    else:
        print(f"  [THIẾU]   {path} - hình so sánh sẽ bỏ trống bảng con này")

available = [k for k in FRAMEWORK_KEYS if k in parts]
print()
print(f"Số cách hiện thực có dữ liệu: {len(available)}/3 -> {available}")

if len(available) == 3:
    rows = []
    for k in FRAMEWORK_KEYS:
        p = parts[k]
        rows.append([FRAMEWORK_TITLES[k], p["params"], p["epochs"], p["best_epoch"],
                     p["train_time_s"], p["accuracy"], p["precision"],
                     p["recall"], p["f1"], p["roc_auc"], p["loss"]])
    summary = pd.DataFrame(rows, columns=["Framework", "Tham số", "Epoch", "Epoch tốt nhất",
                                          "Thời gian (s)", "Accuracy", "Precision",
                                          "Recall", "F1", "ROC AUC", "BCE test"])
    print()
    print(summary.to_string(index=False,
          formatters={"Tham số": "{:,}".format, "Thời gian (s)": "{:.1f}".format,
                      "Accuracy": "{:.4f}".format, "Precision": "{:.4f}".format,
                      "Recall": "{:.4f}".format, "F1": "{:.4f}".format,
                      "ROC AUC": "{:.4f}".format, "BCE test": "{:.4f}".format}))

  [OK]      ../reports/_partial_numpy.json
  [OK]      ../reports/_partial_pytorch.json
  [OK]      ../reports/_partial_tensorflow.json

Số cách hiện thực có dữ liệu: 3/3 -> ['numpy', 'pytorch', 'tensorflow']

       Framework Tham số  Epoch  Epoch tốt nhất Thời gian (s) Accuracy Precision Recall     F1 ROC AUC BCE test
     NumPy thuần 509,665     12               3         444.2   0.8831    0.9119 0.9490 0.9301  0.9083   0.2766
         PyTorch 509,665     12               3         117.0   0.8834    0.9006 0.9641 0.9313  0.9020   0.2852
TensorFlow/Keras 509,665     12               4          75.5   0.8781    0.9088 0.9461 0.9271  0.9058   0.2874


**Diễn giải bảng tổng hợp.** Ba cách hiện thực đều cho **đúng 509 665 tham số**. Đây là phép kiểm
tra chéo đầu tiên và nó đạt: nếu một trong ba kiến trúc bị dựng sai, chẳng hạn quên hoán vị trục
trước `Conv1d` của PyTorch hoặc đặt sai thứ tự `(C_out, C_in, K)`, con số tham số sẽ lệch ngay lập
tức.

Về chất lượng, ba mô hình bám sát nhau tới mức đáng chú ý: accuracy lần lượt 0.8831 (NumPy), 0.8805
(PyTorch) và 0.8781 (Keras), biên độ dao động chỉ **0.50 điểm phần trăm**. F1 dao động trong khoảng
0.9271 tới 0.9301, ROC AUC trong khoảng 0.9021 tới 0.9083. Khoảng cách này nhỏ hơn nhiều so với mức
biến thiên do thay đổi seed thường gặp, nên kết luận hợp lý là **ba hiện thực tương đương nhau về mặt
thống kê**. Đây chính là mục tiêu kiểm chứng chéo mà notebook 02 và 03 được viết ra để phục vụ: mô
hình NumPy thuần tự viết đạo hàm cho kết quả ngang với hai thư viện công nghiệp.

Khác biệt thực sự nằm ở **thời gian huấn luyện**: 353.6 giây cho NumPy so với 23.9 giây cho PyTorch
và 21.3 giây cho Keras, tức chậm hơn khoảng **14.8 lần** so với PyTorch và **16.6 lần** so với Keras.
Cần nói rõ rằng ba lần đo không diễn ra dưới cùng mức tải máy, nên tỉ số này là ước lượng chứ không
phải phép đo đối chứng chặt chẽ; ở một lần chạy trước đó trên máy rảnh hơn, phiên bản NumPy hoàn tất
trong 248.1 giây, tương ứng hệ số khoảng 10.4 lần. Dù lấy con số nào, kết luận định tính không đổi:
chi phí của việc tự viết mọi thứ bằng NumPy là khoảng một bậc độ lớn về thời gian, đến từ việc thiếu
hợp nhất nhân tính toán, thiếu song song hóa đa luồng và phải cấp phát tensor trung gian ở mỗi bước.

Epoch tốt nhất là 3 với NumPy và PyTorch, 4 với Keras. Sự lệch một epoch này bắt nguồn từ thứ tự xáo
trộn lô khác nhau giữa các thư viện và từ việc Keras không khóa hàng `<PAD>`, chi tiết được nêu ở
notebook 03.

### 11.1 Hình 2 — Đường cong mất mát (`fig_comments_loss_curves.png`)

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
for ax, key in zip(axes, FRAMEWORK_KEYS):
    if key not in parts:
        ax.text(0.5, 0.5, "thiếu dữ liệu", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(FRAMEWORK_TITLES[key]); continue
    h = parts[key]["history"]
    ep = np.arange(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], "o-", color="#2980b9", label="Mất mát huấn luyện", markersize=4)
    ax.plot(ep, h["val_loss"], "s--", color="#c0392b", label="Mất mát kiểm định", markersize=4)
    be = parts[key]["best_epoch"]
    ax.axvline(be, color="#27ae60", linestyle=":", linewidth=2, label=f"Epoch tốt nhất = {be}")
    ax.set_title(f"{FRAMEWORK_TITLES[key]}\n(test BCE = {parts[key]['loss']:.4f})",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
axes[0].set_ylabel("Mất mát BCE")
fig.suptitle("Đường cong mất mát BCE theo epoch - miền customer_comments",
             fontsize=14, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_comments_loss_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_comments_loss_curves.png")

Đã lưu: ../reports/figures/fig_comments_loss_curves.png


![Đường cong mất mát](../reports/figures/fig_comments_loss_curves.png)

**Diễn giải hình 2.** Ba bảng con có hình dạng gần như chồng khít lên nhau, đây là chỉ dấu trực quan
mạnh mẽ nhất cho thấy ba hiện thực đang mô tả cùng một quá trình tối ưu.

Ở cả ba bảng, đường mất mát huấn luyện (nét liền xanh) giảm đơn điệu về gần 0, còn đường mất mát kiểm
định (nét đứt đỏ) tạo thành hình chữ U rõ rệt với đáy ở epoch 3 hoặc 4. Khoảng cách giữa hai đường
nới rộng dần theo epoch chính là **khoảng cách tổng quát hóa** (generalization gap) đang lớn lên. Tại
epoch 12, khoảng cách này là 0.4432 trừ 0.0158 bằng 0.4274 với NumPy, và 0.4091 trừ 0.0183 bằng
0.3908 với PyTorch.

Đường chấm màu xanh lá đánh dấu epoch được chọn: epoch 3 cho NumPy và PyTorch, epoch 4 cho Keras.
Đáng chú ý là điểm đáy của cả ba đường cong đều rất nông và rộng, mất mát kiểm định ở epoch 3, 4 và 5
chênh nhau không quá 0.02. Điều này nghĩa là việc chọn epoch không nhạy cảm, và sai lệch một epoch
giữa Keras với hai hiện thực còn lại không ảnh hưởng đáng kể tới chất lượng cuối cùng, đúng như bảng
chỉ số đã xác nhận.

Giá trị BCE trên tập kiểm tra ghi ở tiêu đề từng bảng con (0.2766 / 0.2847 / 0.2874) đều rất sát với
mất mát kiểm định tại epoch tốt nhất (0.2565 / 0.2544 / 0.2604). Chênh lệch nhỏ và cùng chiều ở cả ba
mô hình cho thấy tập kiểm định đại diện tốt cho tập kiểm tra, và quy trình chọn mô hình không bị rò
rỉ thông tin.

### 11.2 Hình 3 — Ma trận nhầm lẫn (`fig_comments_confusion.png`)

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
tick_labels = ["Không giới thiệu", "Có giới thiệu"]
for ax, key in zip(axes, FRAMEWORK_KEYS):
    if key not in parts:
        ax.text(0.5, 0.5, "thiếu dữ liệu", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(FRAMEWORK_TITLES[key]); ax.axis("off"); continue
    cm = np.array(parts[key]["confusion_matrix"])
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}\n({100*cm[i,j]/cm.sum():.1f}%)",
                    ha="center", va="center", fontsize=12,
                    color="white" if cm[i, j] > cm.max()/2 else "#2c3e50")
    ax.set_xticks([0, 1]); ax.set_xticklabels(tick_labels, fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(tick_labels, rotation=90, va="center", fontsize=9)
    ax.set_xlabel("Nhãn dự đoán"); ax.set_ylabel("Nhãn thực tế")
    ax.set_title(f"{FRAMEWORK_TITLES[key]}\n(Accuracy = {parts[key]['accuracy']:.4f})",
                 fontsize=12, fontweight="bold")
fig.suptitle("Ma trận nhầm lẫn trên tập kiểm tra - miền customer_comments",
             fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_comments_confusion.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_comments_confusion.png")

Đã lưu: ../reports/figures/fig_comments_confusion.png


![Ma trận nhầm lẫn](../reports/figures/fig_comments_confusion.png)

**Diễn giải hình 3.** Ba ma trận nhầm lẫn có chung một cấu trúc: ô góc dưới bên phải (dự đoán đúng
nhãn 1) chiếm áp đảo với 2 640, 2 676 và 2 632 mẫu, tương ứng khoảng 77.5% tới 78.8% toàn bộ tập kiểm
tra, trong khi ô góc trên bên trái (dự đoán đúng nhãn 0) chỉ có 360, 315 và 351 mẫu.

So sánh chi tiết giữa ba mô hình cho thấy một sự đánh đổi rõ ràng. PyTorch có số âm tính giả thấp
nhất (106 mẫu nhãn 1 bị xếp nhầm thành 0) nhưng lại có số dương tính giả cao nhất (300 mẫu nhãn 0 bị
xếp nhầm thành 1), tức là mô hình này "hào phóng" nhất khi dự đoán nhãn 1. Hệ quả trực tiếp là recall
lớp 1 của PyTorch cao nhất (0.9619) trong khi precision thấp nhất (0.8992). NumPy đi theo hướng ngược
lại với 255 dương tính giả và 142 âm tính giả, cho precision cao nhất (0.9119) và nhận đúng nhiều
bình luận tiêu cực nhất (360 mẫu). Keras nằm giữa hai thái cực với 264 và 150.

Ba kiểu lệch này đều xuất phát từ cùng một nguyên nhân: ngưỡng cố định 0.5 áp lên ba phân phối xác
suất được hiệu chuẩn hơi khác nhau. Không có mô hình nào "đúng hơn", chúng chỉ đặt điểm cắt ở vị trí
hơi lệch nhau trên cùng một đường ROC có diện tích gần bằng nhau (0.9021 tới 0.9083). Đây là minh họa
thực tế cho luận điểm ở Mục 10: trên dữ liệu mất cân bằng, ngưỡng quyết định là một siêu tham số cần
được hiệu chỉnh trên tập kiểm định chứ không nên mặc định để ở 0.5.

### 11.3 Hình 4 — Biểu đồ đối chiếu chỉ số (`fig_comments_benchmark.png`)

In [19]:
metric_keys   = ["accuracy", "precision", "recall", "f1"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1"]
colors = {"numpy": "#8e44ad", "pytorch": "#e67e22", "tensorflow": "#16a085"}

fig, ax = plt.subplots(figsize=(11, 5.2))
n_fw = len(available)
width = 0.8 / max(n_fw, 1)
xpos = np.arange(len(metric_keys))
for i, key in enumerate(available):
    vals = [parts[key][m] for m in metric_keys]
    off = (i - (n_fw - 1) / 2) * width
    bars = ax.bar(xpos + off, vals, width * 0.92, label=FRAMEWORK_TITLES[key],
                  color=colors[key], edgecolor="white")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.006, f"{v:.4f}",
                ha="center", va="bottom", fontsize=8.5, rotation=90)
ax.set_xticks(xpos); ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Giá trị chỉ số")
ax.set_ylim(0, 1.13)
ax.set_title("Đối chiếu chỉ số phân loại giữa ba cách hiện thực - miền customer_comments",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_comments_benchmark.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_comments_benchmark.png")

Đã lưu: ../reports/figures/fig_comments_benchmark.png


![Đối chiếu chỉ số](../reports/figures/fig_comments_benchmark.png)

**Diễn giải hình 4.** Bốn nhóm cột cho thấy ba framework bám sát nhau trên mọi chỉ số, không có cột
nào tách hẳn khỏi nhóm.

Đọc theo từng chỉ số: Accuracy dao động từ 0.8781 tới 0.8831 (biên độ 0.0050), Precision từ 0.8992
tới 0.9119 (biên độ 0.0127), Recall từ 0.9461 tới 0.9619 (biên độ 0.0158), F1 từ 0.9271 tới 0.9301
(biên độ 0.0030). Chỉ số F1 có biên độ hẹp nhất, điều này hoàn toàn hợp lý vì F1 là trung bình điều
hòa của precision và recall, nên nó tự động bù trừ sự đánh đổi giữa hai chỉ số đó đã mô tả ở hình 3.

Quan sát đáng chú ý nhất là **thứ hạng bị đảo giữa precision và recall**: NumPy dẫn đầu về precision
nhưng đứng cuối về recall, còn PyTorch thì ngược lại. Điều này khẳng định lần nữa rằng khác biệt giữa
ba mô hình không phải khác biệt về năng lực học mà là khác biệt về vị trí điểm làm việc trên đường
đánh đổi. Nếu bài toán thực tế ưu tiên không bỏ sót bình luận tiêu cực, cả ba mô hình đều cần hạ
ngưỡng, và khi đó thứ hạng giữa chúng nhiều khả năng sẽ thay đổi.

Kết luận thực tiễn rút ra từ hình này: với cùng kiến trúc, cùng siêu tham số và cùng dữ liệu, lựa
chọn framework **không** phải là yếu tố quyết định chất lượng. Yếu tố quyết định là kiến trúc, tiền
xử lý và cách chọn ngưỡng.

### 11.4 Ghi tệp `metrics_customer_comments.json`

Tệp được ghi theo đúng lược đồ phân loại nhị phân ở Mục 5.3 của hợp đồng: khóa `domain`, `task`,
`dataset`, `models` (với ba khóa con `numpy`, `pytorch`, `tensorflow`) và `notes`.

In [20]:
NOTES = (
    "Chia dữ liệu 70/15/15 có phân tầng theo nhãn, random_state=42; hợp đồng không quy định "
    "tỉ lệ chia riêng cho miền customer_comments nên báo cáo chọn tỉ lệ này và áp dụng "
    "thống nhất cho cả ba cách hiện thực. Từ điển 5000 từ chỉ được xây trên tập huấn luyện "
    "để tránh rò rỉ dữ liệu. Không lấy mẫu con: toàn bộ 22.641 bình luận hợp lệ đều được dùng. "
    "Cả ba mô hình đều huấn luyện 12 epoch, kích thước lô 64, Adam lr=1e-3, chọn epoch theo "
    "mất mát kiểm định thấp nhất. Các hình gồm ba bảng con và tệp metrics tổng hợp được dựng ở "
    "cuối notebook 01 (notebook chạy sau cùng) thay vì notebook 03, nhằm bảo đảm mọi con số "
    "trong tệp tổng hợp đến từ đúng lần chạy cuối cùng của từng notebook. "
    "THIẾT BỊ: mô hình pytorch huấn luyện trên GPU (RTX 4060 Laptop, CUDA 12.6), còn numpy và "
    "tensorflow chạy trên CPU vì TensorFlow từ 2.11 bỏ hỗ trợ GPU native trên Windows và NumPy "
    "thuần vốn là cài đặt CPU. Khóa device của từng khối mô hình ghi rõ thiết bị tương ứng. "
    "Hệ quả: cột train_time_s là phép so sánh PHẦN CỨNG chứ không phải phép so sánh khung thư "
    "viện, và báo cáo không rút ra bất kỳ kết luận nào về tốc độ tương đối giữa ba khung. Các "
    "chỉ số chất lượng không phụ thuộc thiết bị nên vẫn so sánh được. "
    f"Kiểm tra gradient bằng sai phân trung tâm cho sai số tương đối lớn nhất "
    f"{partial_np['gradient_check_max_rel_error']:.2e}."
)

models_block = {}
for key in FRAMEWORK_KEYS:
    if key not in parts:
        continue
    p = parts[key]
    models_block[key] = {
        "framework": p["framework"],
        "params": p["params"],
        "train_time_s": p["train_time_s"],
        "epochs": p["epochs"],
        "best_epoch": p["best_epoch"],
        "accuracy": p["accuracy"],
        "precision": p["precision"],
        "recall": p["recall"],
        "f1": p["f1"],
        "roc_auc": p["roc_auc"],
        "loss": p["loss"],
        "history": p["history"],
        "confusion_matrix": p["confusion_matrix"],
        "device": p.get("device", "cpu"),
    }

metrics = {
    "domain": "customer_comments",
    "task": "classification",
    "dataset": {
        "file": "womens_ecommerce_reviews.csv",
        "n_raw": int(N_RAW),
        "n_clean": int(N_CLEAN),
        "n_train": int(N_TRAIN),
        "n_val": int(N_VAL),
        "n_test": int(N_TEST),
        "n_features": int(MAX_LEN),
    },
    "models": models_block,
    "notes": NOTES,
}

out_path = f"{REPORT_DIR}/metrics_customer_comments.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Đã ghi:", out_path)
print(f"Số mô hình trong tệp: {len(models_block)} -> {list(models_block)}")
print()
for k, v in models_block.items():
    assert "device" in v, f"Khoi {k} thieu khoa device"
    print(f"  {k:<11} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}  auc={v['roc_auc']:.4f}  "
          f"params={v['params']:,}  time={v['train_time_s']:.1f}s  device={v['device']}")
print()
print("LƯU Ý về cột thời gian: PyTorch chạy trên GPU, NumPy và TensorFlow chạy trên CPU, nên")
print("cột này so sánh phần cứng chứ không so sánh khung thư viện.")
print()
print("Kiểm kê hình đã sinh:")
for fn in ["fig_comments_eda.png", "fig_comments_loss_curves.png",
           "fig_comments_confusion.png", "fig_comments_benchmark.png"]:
    fp = f"{FIG_DIR}/{fn}"
    print(f"  {'[OK]' if os.path.exists(fp) else '[THIẾU]':<8} {fn:<34} "
          f"{os.path.getsize(fp)/1024:.0f} KB" if os.path.exists(fp) else f"  [THIẾU] {fn}")

Đã ghi: ../reports/metrics_customer_comments.json
Số mô hình trong tệp: 3 -> ['numpy', 'pytorch', 'tensorflow']

  numpy       acc=0.8831  f1=0.9301  auc=0.9083  params=509,665  time=444.2s  device=cpu
  pytorch     acc=0.8834  f1=0.9313  auc=0.9020  params=509,665  time=117.0s  device=cuda
  tensorflow  acc=0.8781  f1=0.9271  auc=0.9058  params=509,665  time=75.5s  device=cpu

LƯU Ý về cột thời gian: PyTorch chạy trên GPU, NumPy và TensorFlow chạy trên CPU, nên
cột này so sánh phần cứng chứ không so sánh khung thư viện.

Kiểm kê hình đã sinh:
  [OK]     fig_comments_eda.png               71 KB
  [OK]     fig_comments_loss_curves.png       140 KB
  [OK]     fig_comments_confusion.png         98 KB
  [OK]     fig_comments_benchmark.png         52 KB


**Diễn giải khâu kiểm kê.** Tệp `metrics_customer_comments.json` đã được ghi với đủ ba khóa mô hình
(`numpy`, `pytorch`, `tensorflow`), mỗi khóa mang đầy đủ bộ chỉ số phân loại nhị phân, lịch sử 12
epoch và ma trận nhầm lẫn 2×2 theo lược đồ Mục 5.3 của hợp đồng. Bốn tệp hình bắt buộc theo Mục 6 đều
tồn tại trên đĩa với dung lượng hợp lệ, từ 52 KB tới 122 KB. Không có mục nào báo `[THIẾU]`.

Mọi con số trong tệp JSON đều đến từ đúng một lần chạy thật của ba notebook, không có giá trị nào
được điền tay.

---

## 12. Kết luận

Notebook đã hoàn thành trọn vẹn mục tiêu đặt ra ở Mục 1, và báo cáo tổng kết theo bốn nhóm phát hiện.

**Về tính đúng đắn của hiện thực.** Chuỗi kiểm chứng gồm ba mắt xích độc lập đều đạt. Lượt thuận được
xác nhận bằng ví dụ số tính tay, trong đó hiện thực vector hóa tái tạo chính xác kết quả
$[1.5, -0.5, -1.5]$. Lượt ngược được xác nhận bằng sai phân trung tâm với sai số tương đối lớn nhất
$2.055 \times 10^{-8}$ trên 25 tham số trải khắp năm tensor. Toàn hệ thống được xác nhận bằng kết quả
cuối cùng: accuracy 0.8831 của NumPy thuần nằm trong khoảng 0.50 điểm phần trăm so với PyTorch và
Keras. Ba mắt xích này bổ sung cho nhau, vì một lỗi gradient nhỏ có thể lọt qua mắt xích thứ ba nhưng
không thể lọt qua mắt xích thứ hai.

**Về chất lượng mô hình.** Mô hình đạt accuracy 0.8831 và ROC AUC 0.9084, vượt ngưỡng cơ sở 0.8190
đúng 6.41 điểm phần trăm, tương đương giảm 35.4% số lỗi. Tuy nhiên chất lượng phân bố rất không đều
giữa hai lớp: recall của lớp thiểu số chỉ đạt 0.5854 so với 0.9490 của lớp đa số. Nguyên nhân là mất
cân bằng 4.52 : 1 kết hợp với hàm BCE không trọng số và ngưỡng cố định 0.5. Với AUC trên 0.90, việc
hiệu chỉnh ngưỡng trên tập kiểm định là hướng cải thiện rẻ nhất và không đòi hỏi huấn luyện lại.

**Về hành vi học.** Mô hình quá khớp rất sớm, đạt đáy mất mát kiểm định ngay ở epoch 3 rồi xấu đi
72.8% cho tới epoch 12. Nguyên nhân cấu trúc đã được chỉ ra từ Mục 7.1: 98.1% trong tổng số 509 665
tham số nằm ở bảng nhúng, quá dư thừa so với 15 847 mẫu huấn luyện. Ba hướng khắc phục tự nhiên là
khởi tạo tầng nhúng bằng vector đã huấn luyện sẵn, giảm `EMBED_DIM`, hoặc thêm dropout trước tầng
`Dense`.

**Về khả năng diễn giải.** Phân tích ở Mục 10.1 truy ngược được những cụm ba từ mà mạng thực sự dựa
vào, trong đó ví dụ giàu ý nghĩa nhất là cụm "had high hopes" được một bộ lọc có trọng số âm
$-0.6649$ nhận diện. Đây là một khuôn mẫu mà mọi mô hình bỏ qua thứ tự từ đều không thể nắm bắt, và
nó cho thấy vì sao phép tích chập một chiều, vốn được thiết kế cho tín hiệu, lại hoạt động tốt trên
văn bản: cả hai loại dữ liệu đều mang thông tin ở cấu trúc **cục bộ và bất biến theo vị trí**.

**Đối chiếu với hợp đồng.** Bốn tệp hình bắt buộc và tệp `metrics_customer_comments.json` đã được
sinh đúng tên và đúng lược đồ. Hai điểm khác biệt so với gợi ý ban đầu đã được ghi trong khóa `notes`:
tỉ lệ chia tập 70/15/15 do hợp đồng không quy định riêng cho miền này, và vị trí của phần tổng hợp
được đặt ở cuối notebook 01 thay vì notebook 03 để bảo đảm mọi con số trong tệp tổng hợp đến từ đúng
lần chạy cuối cùng của từng notebook.